In [1]:
import heapq
from collections import defaultdict, deque

# 1) Grid Traversal

**In plain English.** A grid is a graph: each cell is a node and its 4 neighbours are its
edges. "Islands", "regions" and "spreading" are all about visiting every cell you can
reach from a start, marking cells so you never visit them twice.

> **Analogy.** Spilling paint on a tiled floor: it spreads to every touching tile of the
> same kind and stops at walls. Counting islands = counting how many separate spills you
> need to cover all the land.

Every graph question has two steps: **model it** (what is a node, what is an edge, is it
weighted or directed?) and **pick the traversal**:

| Need | Algorithm | Cost |
|---|---|---|
| reachability, components, flood fill | DFS or BFS | O(V + E) |
| fewest steps, unweighted | BFS (multi-source if several starts) | O(V + E) |
| ordering with prerequisites, cycle in a directed graph | topological sort (Kahn) | O(V + E) |
| merging groups, cycle in an undirected graph | union–find | ~O(α(n)) per op |
| shortest path, non-negative weights | Dijkstra | O(E log V) |
| shortest path with ≤ k edges | Bellman–Ford (k rounds) | O(k · E) |
| connect everything at minimum cost | Prim / Kruskal (MST) | O(E log V) |

```python
def dfs(r, c):
    if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != target:
        return
    grid[r][c] = visited_mark          # mark before recursing
    for dr, dc in DIRS:
        dfs(r + dr, c + dc)
```

**Recognition signals**

- "island", "region", "connected", "flood fill", "surrounded"
- "minimum minutes / steps / distance" on a grid → BFS
- "distance to the nearest X" from many X's → multi-source BFS
- "water flows to the ocean" → reverse the direction and start from the target

**Common bugs**

- Marking visited when *popping* instead of when *pushing* → the same cell is queued many
  times.
- Forgetting the bounds check before indexing.
- Recursion depth: a 300×300 grid can need 90 000 frames — use an explicit stack or BFS.
- Flood fill with `new_color == old_color` loops forever.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Grid DP | values depend on neighbours through a recurrence; no "visited" spreading |
| Matrix traversal (spiral, rotate) | fixed visiting order, not reachability |
| Backtracking on a grid (Word Search) | must *unmark* cells after exploring a path; flood fill never unmarks |

**Learning path.** Count components (Islands) → size of components → recolouring → local
counting (Perimeter) → "safe from the border" inversion (Surrounded Regions) → reverse
flow from two targets (Pacific Atlantic). Then BFS for distance: multi-source spreading
(Oranges, Walls and Gates, 01 Matrix) → single-source shortest path (binary matrix, maze)
→ DFS + BFS combined (Shortest Bridge).

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

## 1.1 Connected regions (DFS)

Scan every cell; each unvisited start cell begins one traversal that marks its whole
region. Number of starts = number of regions. DFS or BFS both work — only the marking
matters.

### 1.1.1 [200. Number of Islands](https://leetcode.com/problems/number-of-islands/) — ★★★ · Medium

**Problem.** Count islands of `"1"` cells (4-directionally connected) in a grid.  
**Signal.** Count connected components → start a traversal from every unvisited land cell.  
*New here: Counting components: one traversal per unvisited start.*

**Fine print.** Ask before coding:
- 4-directional only, no diagonals? → Yes.
- Cells are the strings `"1"`/`"0"`? → Yes on LeetCode.
- Can I modify the grid to mark visited cells? → Usually yes; otherwise use a `visited` set.

> 💡 **Key insight.** Each traversal sinks one entire island, so the number of traversals started is the number of islands.

**Brute force →** Union–find over all land cells → also O(rows · cols), more code. DFS/BFS with marking is the standard.

**Steps**
1. Scan every cell.
2. On unvisited land, count one island.
3. Sink the whole island (mark every reachable land cell as water).
4. Continue scanning.

**Complexity**

- **Time:** O(rows · cols) — each cell is visited a constant number of times.
- **Space:** O(rows · cols) worst case for the stack.

In [3]:
def num_islands(grid: list[list[str]]) -> int:
    """Number of 4-connected land regions."""
    rows, cols = len(grid), len(grid[0])

    def sink(r: int, c: int) -> None:
        # iterative DFS; mark on push so no cell is queued twice
        stack = [(r, c)]
        grid[r][c] = "0"
        while stack:
            r, c = stack.pop()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1":
                    grid[nr][nc] = "0"
                    stack.append((nr, nc))

    count = 0
    for r in range(rows):
        for c in range(cols):
            # unvisited land starts a new island
            if grid[r][c] == "1":
                count += 1
                sink(r, c)
    return count


g = [list("11000"), list("11000"), list("00100"), list("00011")]
assert num_islands(g) == 3
assert num_islands([list("11110"), list("11010"), list("11000"), list("00000")]) == 1
print("ok")

ok


### 1.1.2 [695. Max Area of Island](https://leetcode.com/problems/max-area-of-island/) — ★★ · Medium

**Problem.** Area of the largest island (cells of value 1).  
**Signal.** Components with a size → DFS returning the count.  
*New here: A traversal that *returns* a value (the size).*

**Fine print.** Ask before coding:
- No island? → Return 0.
- 4-directional? → Yes.

> 💡 **Key insight.** An island's area is 1 for this cell plus the areas reachable from its neighbours, as long as each cell is counted once.

**Brute force →** Count each island with a separate visited set → same idea, more memory. Marking the grid itself is enough.

**Steps**
1. `area(r, c)` returns 0 outside the grid or on water.
2. Mark the cell as water, return `1 + sum of neighbour areas`.
3. Take the max over all cells.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(rows · cols) recursion in the worst case.

In [4]:
def max_area_of_island(grid: list[list[int]]) -> int:
    """Largest island area."""
    rows, cols = len(grid), len(grid[0])

    def area(r: int, c: int) -> int:
        # outside, water, or already counted
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != 1:
            return 0
        grid[r][c] = 0
        # this cell plus everything reachable from it
        return 1 + sum(area(r + dr, c + dc) for dr, dc in DIRS)

    return max(area(r, c) for r in range(rows) for c in range(cols))


g = [
    [0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0],
    [0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
]
assert max_area_of_island(g) == 6
assert max_area_of_island([[0, 0]]) == 0
print("ok")

ok


### 1.1.3 [733. Flood Fill](https://leetcode.com/problems/flood-fill/) — ★★ · Easy

**Problem.** Recolour the connected region of the start pixel's colour.  
**Signal.** Paint-bucket = DFS from one cell.  
*New here: Recolouring doubles as the visited mark — with one edge case.*

**Fine print.** Ask before coding:
- Does the new colour equal the old one? → It can, so return early or the DFS loops.
- 4-directional? → Yes.

> 💡 **Key insight.** Painting a cell both changes it and marks it visited, unless the new colour equals the old one.

**Brute force →** Separate visited set → O(rows · cols) extra; recolouring in place does both jobs.

**Steps**
1. Remember the old colour; if it equals the new colour, return.
2. DFS from the start, recolouring cells that still have the old colour.
3. Recurse into the 4 neighbours.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(rows · cols) recursion worst case.

⚠️ **Trap.** Without the `old == color` check, nothing ever looks visited and the DFS never ends.

In [5]:
def flood_fill(image: list[list[int]], sr: int, sc: int, color: int) -> list[list[int]]:
    """Image after flood fill from (sr, sc)."""
    old = image[sr][sc]
    # same colour: nothing to do (and the DFS would never stop)
    if old == color:
        return image
    rows, cols = len(image), len(image[0])

    def fill(r: int, c: int) -> None:
        if 0 <= r < rows and 0 <= c < cols and image[r][c] == old:
            # painting also marks the cell as visited
            image[r][c] = color
            for dr, dc in DIRS:
                fill(r + dr, c + dc)

    fill(sr, sc)
    return image


assert flood_fill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2) == [
    [2, 2, 2],
    [2, 2, 0],
    [2, 0, 1],
]
assert flood_fill([[0, 0, 0], [0, 0, 0]], 0, 0, 0) == [[0, 0, 0], [0, 0, 0]]
print("ok")

ok


### 1.1.4 [463. Island Perimeter](https://leetcode.com/problems/island-perimeter/) — ★ · Easy

**Problem.** Perimeter of the single island in a grid.  
**Signal.** Count edges: each land cell adds 4, each shared edge removes 2.  
*New here: Some grid questions need no traversal — just local counting.*

**Fine print.** Ask before coding:
- Exactly one island and no lakes inside? → Yes.
- Count edges touching water or the border? → Yes.

> 💡 **Key insight.** Every land cell contributes 4 sides, and every pair of touching land cells hides 2 of them.

**Brute force →** DFS counting water/border neighbours per cell → also O(rows · cols), more code.

**Steps**
1. For every land cell, add 4.
2. If the cell above is land, subtract 2.
3. If the cell to the left is land, subtract 2.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(1).

In [6]:
def island_perimeter(grid: list[list[int]]) -> int:
    """Perimeter of the island."""
    total = 0
    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell:
                total += 4
                # a shared edge hides one side of each of the two cells
                if r and grid[r - 1][c]:
                    total -= 2
                if c and row[c - 1]:
                    total -= 2
    return total


assert island_perimeter([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]]) == 16
assert island_perimeter([[1]]) == 4
print("ok")

ok


### 1.1.5 [130. Surrounded Regions](https://leetcode.com/problems/surrounded-regions/) — ★★ · Medium

**Problem.** Flip every `O` region not connected to the border into `X`.  
**Signal.** Invert the question: find what is *safe* (border-connected) and flip the rest.  
*New here: Inverting the question: start from the border, not from each region.*

**Fine print.** Ask before coding:
- Is an `O` on the border ever captured? → No, nor anything connected to it.
- Modify the board in place? → Yes.

> 💡 **Key insight.** A region survives iff it touches the border, so mark everything reachable from border `O`s and flip all other `O`s.

**Brute force →** For each region, DFS and check whether it touches the border → still O(rows · cols) but needs bookkeeping per region. Border-first is simpler.

**Steps**
1. DFS from every border `O`, marking reached cells `#`.
2. Every remaining `O` is surrounded → `X`.
3. Every `#` → back to `O`.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(rows · cols) recursion worst case.

In [7]:
def solve_surrounded(board: list[list[str]]) -> None:
    """Capture surrounded regions in place."""
    rows, cols = len(board), len(board[0])

    def mark(r: int, c: int) -> None:
        # "#" = connected to the border, therefore safe
        if 0 <= r < rows and 0 <= c < cols and board[r][c] == "O":
            board[r][c] = "#"
            for dr, dc in DIRS:
                mark(r + dr, c + dc)

    # start only from border cells
    for r in range(rows):
        mark(r, 0)
        mark(r, cols - 1)
    for c in range(cols):
        mark(0, c)
        mark(rows - 1, c)
    # safe cells go back to O, everything else is captured
    for r in range(rows):
        for c in range(cols):
            board[r][c] = "O" if board[r][c] == "#" else "X"


b = [list("XXXX"), list("XOOX"), list("XXOX"), list("XOXX")]
solve_surrounded(b)
assert b == [list("XXXX"), list("XXXX"), list("XXXX"), list("XOXX")]
print(["".join(r) for r in b])

['XXXX', 'XXXX', 'XXXX', 'XOXX']


### 1.1.6 [417. Pacific Atlantic Water Flow](https://leetcode.com/problems/pacific-atlantic-water-flow/) — ★★★ · Medium

**Problem.** Cells from which water can flow (to equal or lower neighbours) into both the Pacific (top/left edges) and the Atlantic (bottom/right edges).  
**Signal.** Reverse the flow: climb *uphill* from each ocean's border and intersect.  
*New here: Reversing edge direction and intersecting two traversals.*

**Fine print.** Ask before coding:
- Can water flow to a neighbour of equal height? → Yes, `<=`.
- Pacific on the top and left, Atlantic on the bottom and right? → Yes.
- Any output order? → Yes.

> 💡 **Key insight.** "Can this cell reach the ocean downhill?" equals "can the ocean reach this cell uphill?", so two traversals from the borders answer it for all cells at once.

**Brute force →** DFS downhill from every cell → O((rows · cols)²). Two uphill traversals → O(rows · cols).

**Steps**
1. Start set for the Pacific: top row + left column; for the Atlantic: bottom row + right column.
2. From each start set, walk to neighbours with height ≥ the current cell.
3. Collect each ocean's reachable set.
4. Return the intersection.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(rows · cols).

In [8]:
def pacific_atlantic(heights: list[list[int]]) -> list[list[int]]:
    """Cells that drain to both oceans."""
    rows, cols = len(heights), len(heights[0])

    def climb(starts: list[tuple[int, int]]) -> set[tuple[int, int]]:
        """Cells reachable from starts by moving to equal or higher cells."""
        seen = set(starts)
        stack = list(starts)
        while stack:
            r, c = stack.pop()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                # uphill (reverse of water flow)
                if (
                    0 <= nr < rows
                    and 0 <= nc < cols
                    and (nr, nc) not in seen
                    and heights[nr][nc] >= heights[r][c]
                ):
                    seen.add((nr, nc))
                    stack.append((nr, nc))
        return seen

    pacific = climb([(0, c) for c in range(cols)] + [(r, 0) for r in range(rows)])
    atlantic = climb(
        [(rows - 1, c) for c in range(cols)] + [(r, cols - 1) for r in range(rows)]
    )
    # cells reached from both oceans
    return sorted([r, c] for r, c in pacific & atlantic)


h = [
    [1, 2, 2, 3, 5],
    [3, 2, 3, 4, 4],
    [2, 4, 5, 3, 1],
    [6, 7, 1, 4, 5],
    [5, 1, 1, 2, 4],
]
assert pacific_atlantic(h) == [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
print(pacific_atlantic(h))

[[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]


## 1.2 Shortest paths on grids (BFS)

BFS visits cells in order of distance, so the first time it reaches a cell is the shortest
way there. **Multi-source BFS** puts every start in the queue at distance 0 — it computes,
for every cell, the distance to the *nearest* source in one pass. Process the queue level
by level when you need "minutes" or "steps".

### 1.2.1 [994. Rotting Oranges](https://leetcode.com/problems/rotting-oranges/) — ★★ · Medium

**Problem.** Each minute rotten oranges rot their fresh neighbours. Minutes until none are fresh (−1 if impossible)?  
**Signal.** Simultaneous spreading from several sources → multi-source BFS by levels.  
*New here: Multi-source BFS where one level = one minute.*

**Fine print.** Ask before coding:
- No fresh oranges at the start? → Return 0.
- Unreachable fresh oranges? → Return −1.
- 4-directional? → Yes.

> 💡 **Key insight.** All rotten oranges spread at the same time, so putting them all in the queue at minute 0 and expanding level by level simulates the clock exactly.

**Brute force →** Simulate minute by minute rescanning the grid → O((rows · cols)²). Multi-source BFS → O(rows · cols).

**Steps**
1. Queue every rotten orange; count fresh ones.
2. While the queue is non-empty and fresh oranges remain, process one level (= one minute).
3. Each fresh neighbour turns rotten, is queued, and `fresh −= 1`.
4. Return the minutes, or −1 if fresh oranges remain.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(rows · cols).

In [9]:
def oranges_rotting(grid: list[list[int]]) -> int:
    """Minutes until every orange is rotten."""
    rows, cols = len(grid), len(grid[0])
    # every rotten orange starts spreading at minute 0
    queue = deque((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == 2)
    fresh = sum(row.count(1) for row in grid)
    minutes = 0
    while queue and fresh:
        minutes += 1
        # one BFS level = one minute of spreading
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    queue.append((nr, nc))
    return -1 if fresh else minutes


assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
assert oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1
assert oranges_rotting([[0, 2]]) == 0
print("ok")

ok


### 1.2.2 [286. Walls and Gates](https://leetcode.com/problems/walls-and-gates/) — ★★ · Medium · Premium

**Problem.** Fill each empty room (`INF`) with the distance to its nearest gate (`0`); walls are −1.  
**Signal.** Distance to nearest of many sources → multi-source BFS from all gates.  
*New here: Writing distances into the grid itself (the grid is the visited set).*

**Fine print.** Ask before coding:
- In place? → Yes.
- Do unreachable rooms stay `INF`? → Yes.

> 💡 **Key insight.** BFS from all gates together reaches each room first from its nearest gate, so the first distance written is final.

**Brute force →** BFS from every gate separately and keep the minimum → O(gates · rows · cols). One multi-source BFS → O(rows · cols).

**Steps**
1. Queue every gate.
2. Pop a cell; for each neighbour still `INF`, write `dist + 1` and queue it.
3. Walls and filled rooms are never touched again.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(rows · cols).

In [10]:
INF = 2**31 - 1


def walls_and_gates(rooms: list[list[int]]) -> None:
    """Distance to nearest gate, in place."""
    rows, cols = len(rooms), len(rooms[0])
    # all gates start together
    queue = deque((r, c) for r in range(rows) for c in range(cols) if rooms[r][c] == 0)
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            # INF = not reached yet; first arrival is the nearest gate
            if 0 <= nr < rows and 0 <= nc < cols and rooms[nr][nc] == INF:
                rooms[nr][nc] = rooms[r][c] + 1
                queue.append((nr, nc))


rooms = [[INF, -1, 0, INF], [INF, INF, INF, -1], [INF, -1, INF, -1], [0, -1, INF, INF]]
walls_and_gates(rooms)
assert rooms == [[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]
print(rooms)

[[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]


### 1.2.3 [542. 01 Matrix](https://leetcode.com/problems/01-matrix/) — ★★ · Medium

**Problem.** Distance of each cell to the nearest 0.  
**Signal.** Multi-source BFS from every 0.  
*New here: The same pattern with a separate distance grid (−1 = unvisited).*

**Fine print.** Ask before coding:
- Is there at least one 0? → Yes.
- 4-directional, Manhattan steps? → Yes.

> 💡 **Key insight.** Instead of BFS from every 1 toward a 0, BFS once from all 0s outward.

**Brute force →** BFS from each 1 → O((rows · cols)²). Multi-source → O(rows · cols). (Two DP sweeps also work.)

**Steps**
1. `dist` = 0 for zeros, −1 for ones.
2. Queue all zeros.
3. For each popped cell, set unvisited neighbours to `dist + 1` and queue them.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(rows · cols).

In [11]:
def update_matrix(mat: list[list[int]]) -> list[list[int]]:
    """Distance from each cell to the nearest zero."""
    rows, cols = len(mat), len(mat[0])
    # -1 marks "not reached yet"
    dist = [[0 if v == 0 else -1 for v in row] for row in mat]
    queue = deque((r, c) for r in range(rows) for c in range(cols) if mat[r][c] == 0)
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                queue.append((nr, nc))
    return dist


assert update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]) == [
    [0, 0, 0],
    [0, 1, 0],
    [1, 2, 1],
]
print(update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]))

[[0, 0, 0], [0, 1, 0], [1, 2, 1]]


### 1.2.4 [1091. Shortest Path in Binary Matrix](https://leetcode.com/problems/shortest-path-in-binary-matrix/) — ★ · Medium

**Problem.** Shortest 8-directional path of 0-cells from top-left to bottom-right (path length in cells), or −1.  
**Signal.** Unweighted shortest path → BFS with 8 neighbours.  
*New here: Single-source BFS with distance carried in the queue, and 8 directions.*

**Fine print.** Ask before coding:
- 8 directions? → Yes.
- Length counted in cells, not moves? → Cells.
- Blocked start or end? → Return −1.

> 💡 **Key insight.** Every move costs 1, so BFS order is distance order and the first time the target is popped is the shortest path.

**Brute force →** DFS over all paths → exponential. BFS → O(n²).

**Steps**
1. If the start or end is blocked, return −1.
2. Queue `(0, 0, 1)` and mark it.
3. Pop; if it is the target, return its length.
4. Push unvisited open neighbours in all 8 directions with length + 1.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n²).

In [12]:
def shortest_path_binary_matrix(grid: list[list[int]]) -> int:
    """Length of the shortest clear path, or -1."""
    n = len(grid)
    if grid[0][0] or grid[-1][-1]:
        return -1
    # (row, col, path length in cells)
    queue = deque([(0, 0, 1)])
    grid[0][0] = 1
    while queue:
        r, c, d = queue.popleft()
        # BFS order: the first arrival is the shortest
        if (r, c) == (n - 1, n - 1):
            return d
        # all 8 neighbours
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 0:
                    grid[nr][nc] = 1
                    queue.append((nr, nc, d + 1))
    return -1


assert shortest_path_binary_matrix([[0, 1], [1, 0]]) == 2
assert shortest_path_binary_matrix([[0, 0, 0], [1, 1, 0], [1, 1, 0]]) == 4
assert shortest_path_binary_matrix([[1, 0, 0], [1, 1, 0], [1, 1, 0]]) == -1
print("ok")

ok


### 1.2.5 [1926. Nearest Exit from Entrance in Maze](https://leetcode.com/problems/nearest-exit-from-entrance-in-maze/) — ★ · Medium

**Problem.** Fewest steps from the entrance to any border empty cell (not the entrance itself).  
**Signal.** Unweighted shortest path to any of many targets → BFS, stop at the first target.  
*New here: Many targets: stop at the first one BFS reaches.*

**Fine print.** Ask before coding:
- Can the entrance itself be an exit? → No, even if it is on the border.
- No exit? → Return −1.

> 💡 **Key insight.** BFS reaches cells in distance order, so the first border cell popped is the nearest exit.

**Brute force →** BFS to each border cell separately → O(border · cells). One BFS stopping early → O(cells).

**Steps**
1. Queue the entrance with distance 0; mark it.
2. Pop; if it is on the border and not the entrance, return its distance.
3. Push unvisited open neighbours with distance + 1.

**Complexity**

- **Time:** O(rows · cols).
- **Space:** O(rows · cols).

In [13]:
def nearest_exit(maze: list[list[str]], entrance: list[int]) -> int:
    """Steps to the nearest exit, or -1."""
    rows, cols = len(maze), len(maze[0])
    start = tuple(entrance)
    queue = deque([(start, 0)])
    maze[start[0]][start[1]] = "+"
    while queue:
        (r, c), d = queue.popleft()
        # a border cell other than the entrance is an exit
        if (r, c) != start and (r in (0, rows - 1) or c in (0, cols - 1)):
            return d
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] == ".":
                # mark as wall once queued
                maze[nr][nc] = "+"
                queue.append(((nr, nc), d + 1))
    return -1


assert nearest_exit([list("++.+"), list("...+"), list("+++.")], [1, 2]) == 1
assert nearest_exit([list("+++"), list("..."), list("+++")], [1, 0]) == 2
assert nearest_exit([list(".+")], [0, 0]) == -1
print("ok")

ok


### 1.2.6 [934. Shortest Bridge](https://leetcode.com/problems/shortest-bridge/) — ★ · Medium

**Problem.** Exactly two islands; fewest 0-cells to flip to connect them.  
**Signal.** DFS to collect one island, then multi-source BFS outward from all its cells.  
*New here: Chaining two traversals: DFS to find the sources, BFS to measure distance.*

**Fine print.** Ask before coding:
- Exactly two islands? → Yes.
- Count the flipped 0s, not the path length? → Yes.

> 💡 **Key insight.** The shortest bridge is the distance from the *whole* first island to the second, which a multi-source BFS from every cell of the first island measures directly.

**Brute force →** BFS from each cell of island one → O(n⁴). One DFS + one multi-source BFS → O(n²).

**Steps**
1. Find any land cell; DFS its island, marking cells 2 and queueing them.
2. BFS outward level by level over water, marking visited water 2.
3. The first time a neighbour is original land (1), return the level count.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n²).

In [14]:
def shortest_bridge(grid: list[list[int]]) -> int:
    """Zeros to flip to join the two islands."""
    n = len(grid)
    # DFS the first island, marking it 2 and queueing all its cells
    start = next((r, c) for r in range(n) for c in range(n) if grid[r][c])
    queue = deque()
    stack = [start]
    grid[start[0]][start[1]] = 2
    while stack:
        r, c = stack.pop()
        queue.append((r, c))
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 1:
                grid[nr][nc] = 2
                stack.append((nr, nc))
    # BFS over water; each level is one more flipped cell
    steps = 0
    while queue:
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n:
                    # touched the second island
                    if grid[nr][nc] == 1:
                        return steps
                    if grid[nr][nc] == 0:
                        grid[nr][nc] = 2
                        queue.append((nr, nc))
        steps += 1
    return -1


assert shortest_bridge([[0, 1], [1, 0]]) == 1
assert shortest_bridge([[0, 1, 0], [0, 0, 0], [0, 0, 1]]) == 2
g = [
    [1, 1, 1, 1, 1],
    [1, 0, 0, 0, 1],
    [1, 0, 1, 0, 1],
    [1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1],
]
assert shortest_bridge(g) == 1
print("ok")

ok


---
# 2) Graph Traversal

**In plain English.** For general graphs, first turn the input into an adjacency list
(node → neighbours), then run the same DFS/BFS as on grids — but keep a `visited` set,
because graphs can have cycles.

> **Analogy.** Exploring a city's metro: write down for each station which stations it
> connects to, then walk from your start, ticking off stations so you never loop around
> the same circle line forever.

Build it like this: `graph = defaultdict(list)`; for an undirected edge add both
directions.

**Recognition signals**

- "connected", "provinces", "components", "can you reach / visit all"
- "is it a tree" (connected + `n − 1` edges), "bipartite / two groups"
- "clone / deep copy a graph"
- ratios or conversions between things (`a / b = 2`) → weighted graph
- "fewest transformations / moves" between states (words, locks, boards) → BFS on an
  implicit graph

**Common bugs**

- Adding only one direction for undirected edges.
- Marking visited too late (when popping) → exponential duplicates in BFS.
- Forgetting isolated nodes that never appear in the edge list (loop over `range(n)`).
- In BFS over states, generating neighbours that are invalid (dead ends, out of bank).

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Tree DFS | no `visited` needed; a graph needs one |
| Union–find | answers "connected?" while edges keep arriving; DFS/BFS answers it once for a fixed graph |
| Topological sort | for *directed* acyclic dependencies; plain traversal ignores order |
| Dijkstra | edges have different weights; BFS assumes every edge costs 1 |

**Learning path.** Clone (traversal + map) → components on a matrix (Provinces) →
reachability (Keys and Rooms) → components on an edge list → tree check → 2-colouring →
edge direction bookkeeping (Reorder Routes) → weighted path products (Evaluate Division).
Then implicit graphs explored by BFS: Word Ladder → Genetic Mutation → Open the Lock →
Snakes and Ladders → BFS over buses instead of stops.

## 2.1 Adjacency-list traversal

Recipe: build the adjacency list → loop over all nodes → start a traversal from each
unvisited one. The number of starts is the number of components; what you carry along
(a clone map, a colour, an edge flag, a product) is what changes between questions.

### 2.1.1 [133. Clone Graph](https://leetcode.com/problems/clone-graph/) — ★★★ · Medium

**Problem.** Deep-copy a connected undirected graph given one node.  
**Signal.** Traverse the graph while mapping original → clone (the map doubles as `visited`).  
*New here: A map that is both the `visited` set and the result.*

**Fine print.** Ask before coding:
- Undirected and connected? → Yes.
- Can the input node be `None`? → Yes, return `None`.
- Self-loops or repeated edges? → No.

> 💡 **Key insight.** Register a node's clone in the map *before* cloning its neighbours, so a cycle that leads back to it finds the clone instead of recursing forever.

**Brute force →** Copy values first, then search for each neighbour's copy by value → O(V²) and breaks on duplicate values. The original → clone map is O(V + E).

**Steps**
1. If the node is already in the map, return its clone.
2. Create the clone and store it in the map.
3. Clone every neighbour recursively and attach them.
4. Return the clone.

**Complexity**

- **Time:** O(V + E).
- **Space:** O(V).

In [15]:
class Node:
    """Graph node with neighbours."""

    def __init__(self, val: int = 0, neighbors: list | None = None) -> None:
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []


def clone_graph(node: Node | None) -> Node | None:
    """Deep copy of the graph reachable from node."""
    # original -> clone; also serves as the visited set
    clones: dict[Node, Node] = {}

    def dfs(n: Node) -> Node:
        if n in clones:
            return clones[n]
        copy = Node(n.val)
        # register before recursing so cycles find this clone
        clones[n] = copy
        copy.neighbors = [dfs(nb) for nb in n.neighbors]
        return copy

    return dfs(node) if node else None


nodes = {i: Node(i) for i in range(1, 5)}
for a, nbs in {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [1, 3]}.items():
    nodes[a].neighbors = [nodes[b] for b in nbs]
c = clone_graph(nodes[1])
assert c is not nodes[1] and [n.val for n in c.neighbors] == [2, 4]
assert [n.val for n in c.neighbors[0].neighbors] == [1, 3]
print("ok")

ok


### 2.1.2 [547. Number of Provinces](https://leetcode.com/problems/number-of-provinces/) — ★ · Medium

**Problem.** Adjacency matrix of cities; count connected groups.  
**Signal.** Connected components on a matrix-encoded graph.  
*New here: Reading neighbours from an adjacency matrix.*

**Fine print.** Ask before coding:
- Symmetric matrix with 1 on the diagonal? → Yes.
- `n ≤ 200`? → Yes, so O(n²) is fine.

> 💡 **Key insight.** Row `i` of the matrix lists city `i`'s neighbours, so it is the same component count as on an adjacency list.

**Brute force →** Union every connected pair → O(n² α(n)), equally fine. DFS is O(n²) because the matrix has n² entries.

**Steps**
1. `seen` array of size n.
2. For each unseen city, count one province and DFS.
3. The DFS visits every `j` with `is_connected[i][j] == 1` that is unseen.

**Complexity**

- **Time:** O(n²) — every matrix entry is read.
- **Space:** O(n).

In [16]:
def find_circle_num(is_connected: list[list[int]]) -> int:
    """Number of connected components."""
    n = len(is_connected)
    seen = [False] * n

    def dfs(i: int) -> None:
        seen[i] = True
        # row i lists the neighbours of city i
        for j in range(n):
            if is_connected[i][j] and not seen[j]:
                dfs(j)

    count = 0
    for i in range(n):
        # an unseen city starts a new province
        if not seen[i]:
            count += 1
            dfs(i)
    return count


assert find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]) == 2
assert find_circle_num([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 3
print("ok")

ok


### 2.1.3 [841. Keys and Rooms](https://leetcode.com/problems/keys-and-rooms/) — ★ · Medium

**Problem.** Room 0 is open; each room has keys to other rooms. Can you visit all rooms?  
**Signal.** Reachability from node 0.  
*New here: Single-source reachability, answered by comparing counts.*

**Fine print.** Ask before coding:
- Is room 0 open at the start? → Yes.
- Can keys repeat or open rooms you have already visited? → Yes.

> 💡 **Key insight.** Keys are directed edges; you can visit all rooms iff a traversal from room 0 reaches every room.

**Brute force →** Repeatedly loop over unlocked rooms collecting keys until nothing changes → O(n · (n + E)). One DFS → O(n + E).

**Steps**
1. Start with room 0 visited.
2. DFS: for each key in the current room, visit that room if new.
3. Return whether the visited count equals the number of rooms.

**Complexity**

- **Time:** O(n + E) — every key is read once.
- **Space:** O(n).

In [17]:
def can_visit_all_rooms(rooms: list[list[int]]) -> bool:
    """True if every room is reachable from room 0."""
    seen, stack = {0}, [0]
    while stack:
        # each key opens a new room (edge)
        for key in rooms[stack.pop()]:
            if key not in seen:
                seen.add(key)
                stack.append(key)
    return len(seen) == len(rooms)


assert can_visit_all_rooms([[1], [2], [3], []])
assert not can_visit_all_rooms([[1, 3], [3, 0, 1], [2], [0]])
print("ok")

ok


### 2.1.4 [323. Number of Connected Components in an Undirected Graph](https://leetcode.com/problems/number-of-connected-components-in-an-undirected-graph/) — ★★★ · Medium · Premium

**Problem.** `n` nodes and an edge list; count connected components.  
**Signal.** Components → DFS from each unvisited node (or union–find: `n − successful unions`).  
*New here: Building the adjacency list from an edge list, including isolated nodes.*

**Fine print.** Ask before coding:
- Can nodes have no edges? → Yes, each isolated node counts as a component.
- Repeated edges? → No.

> 💡 **Key insight.** Each traversal started from an unvisited node covers exactly one component.

**Brute force →** Union–find also works in O(E α(n)); DFS with an adjacency list is O(V + E).

**Steps**
1. Build an undirected adjacency list.
2. For each node `0..n−1` not yet seen, count a component and traverse it.
3. Return the count.

**Complexity**

- **Time:** O(V + E).
- **Space:** O(V + E).

⚠️ **Trap.** Looping over the edge list instead of `range(n)` misses isolated nodes.

In [18]:
def count_components(n: int, edges: list[list[int]]) -> int:
    """Number of connected components."""
    # undirected: store both directions
    graph = defaultdict(list)
    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)
    seen: set[int] = set()
    count = 0
    # range(n) also covers nodes with no edges
    for start in range(n):
        if start in seen:
            continue
        count += 1
        stack = [start]
        seen.add(start)
        while stack:
            for nb in graph[stack.pop()]:
                if nb not in seen:
                    seen.add(nb)
                    stack.append(nb)
    return count


assert count_components(5, [[0, 1], [1, 2], [3, 4]]) == 2
assert count_components(5, [[0, 1], [1, 2], [2, 3], [3, 4]]) == 1
print("ok")

ok


### 2.1.5 [261. Graph Valid Tree](https://leetcode.com/problems/graph-valid-tree/) — ★★★ · Medium · Premium

**Problem.** Do `n` nodes and these undirected edges form a tree?  
**Signal.** A tree has exactly `n − 1` edges and is connected (equivalently: no cycle).  
*New here: Reducing a property to two cheap checks.*

**Fine print.** Ask before coding:
- Undirected? → Yes.
- Can there be repeated edges or self-loops? → No; a tree needs exactly `n − 1` edges and must be connected.

> 💡 **Key insight.** A graph with n nodes is a tree iff it has exactly `n − 1` edges and is connected — then a cycle is impossible.

**Brute force →** DFS looking for a back edge (with parent tracking) plus a connectivity check → more code. Edge count + one component check is simplest.

**Steps**
1. If `len(edges) != n − 1`, return `False`.
2. Otherwise return whether there is exactly one component.

**Complexity**

- **Time:** O(V + E).
- **Space:** O(V + E).

In [19]:
def valid_tree(n: int, edges: list[list[int]]) -> bool:
    """True if the graph is a tree."""
    # n - 1 edges + connected  =>  no cycle
    return len(edges) == n - 1 and count_components(n, edges) == 1


assert valid_tree(5, [[0, 1], [0, 2], [0, 3], [1, 4]])
assert not valid_tree(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]])
print("ok")

ok


### 2.1.6 [785. Is Graph Bipartite?](https://leetcode.com/problems/is-graph-bipartite/) — ★ · Medium

**Problem.** Can nodes be 2-coloured so every edge joins different colours?  
**Signal.** 2-colouring via BFS/DFS; conflict ⇔ odd cycle.  
*New here: Carrying a label (colour) through a traversal and checking it on every edge.*

**Fine print.** Ask before coding:
- Can the graph be disconnected? → Yes, so start a BFS from every uncoloured node.
- Self-loops or parallel edges? → No.

> 💡 **Key insight.** Once one node's colour is fixed, every neighbour's colour is forced; a contradiction anywhere means the graph is not bipartite.

**Brute force →** Try all 2ⁿ colourings → exponential. Forced colouring by traversal → O(V + E).

**Steps**
1. For each uncoloured node, colour it 0 and traverse its component.
2. Give every uncoloured neighbour the opposite colour.
3. A neighbour that already has the *same* colour → `False`.
4. No conflicts → `True`.

**Complexity**

- **Time:** O(V + E).
- **Space:** O(V).

In [20]:
def is_bipartite(graph: list[list[int]]) -> bool:
    """True if the graph is 2-colourable."""
    color: dict[int, int] = {}
    # every component needs its own start
    for start in range(len(graph)):
        if start in color:
            continue
        color[start] = 0
        stack = [start]
        while stack:
            node = stack.pop()
            for nb in graph[node]:
                # forced to the opposite colour
                if nb not in color:
                    color[nb] = 1 - color[node]
                    stack.append(nb)
                # same colour across an edge: odd cycle
                elif color[nb] == color[node]:
                    return False
    return True


assert not is_bipartite([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]])
assert is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]])
print("ok")

ok


### 2.1.7 [1466. Reorder Routes to Make All Paths Lead to the City Zero](https://leetcode.com/problems/reorder-routes-to-make-all-paths-lead-to-the-city-zero/) — ★ · Medium

**Problem.** Tree of directed roads; minimum number of roads to reverse so every city can reach 0.  
**Signal.** Traverse the undirected tree from 0, counting edges that point away from 0.  
*New here: Storing edge direction as a flag on an undirected adjacency list.*

**Fine print.** Ask before coding:
- Is the underlying graph a tree? → Yes, `n − 1` roads and connected.
- Count only the roads that point away from 0? → Yes.

> 💡 **Key insight.** Walking outward from city 0, every road you traverse in its original direction points away from 0 and must be flipped.

**Brute force →** For each city, check whether it reaches 0 and try flips → exponential. One traversal counting outward edges → O(n).

**Steps**
1. For each road `a → b`, store `(b, 1)` at `a` and `(a, 0)` at `b`.
2. DFS from 0.
3. Each time you cross a neighbour with flag 1, count a flip.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [21]:
def min_reorder(n: int, connections: list[list[int]]) -> int:
    """Number of edges to flip so all reach city 0."""
    # flag 1: the road really goes this way (away from the caller)
    graph = defaultdict(list)
    for a, b in connections:
        graph[a].append((b, 1))
        graph[b].append((a, 0))
    seen, stack, flips = {0}, [0], 0
    while stack:
        node = stack.pop()
        for nb, is_original in graph[node]:
            if nb not in seen:
                seen.add(nb)
                # a road pointing away from city 0 must be reversed
                flips += is_original
                stack.append(nb)
    return flips


assert min_reorder(6, [[0, 1], [1, 3], [2, 3], [4, 0], [4, 5]]) == 3
assert min_reorder(5, [[1, 0], [1, 2], [3, 2], [3, 4]]) == 2
print("ok")

ok


### 2.1.8 [399. Evaluate Division](https://leetcode.com/problems/evaluate-division/) — ★★ · Medium

**Problem.** Given `a / b = k` equations, answer queries `x / y` (or −1).  
**Signal.** Weighted graph: edge `a → b` with weight `k`, `b → a` with `1/k`; a query multiplies weights along a path.  
*New here: Modelling a non-graph statement as a weighted graph.*

**Fine print.** Ask before coding:
- An unknown variable, or no path between them? → Return −1.0.
- `x / x` for a known variable? → 1.0.
- Are the equations consistent? → Yes, guaranteed.

> 💡 **Key insight.** `x / y` = product of the ratios along any path from `x` to `y`, because intermediate variables cancel.

**Brute force →** Derive all ratios by repeated substitution → messy. Path search per query → O(Q · (V + E)); union–find with weights gets near O(1) per query.

**Steps**
1. Add edges `a → b` (k) and `b → a` (1/k).
2. For each query, unknown variables → −1.
3. DFS from `x`, multiplying weights, until `y` is reached.
4. No path → −1.

**Complexity**

- **Time:** O(Q · (V + E)).
- **Space:** O(V + E).

In [22]:
def calc_equation(
    equations: list[list[str]], values: list[float], queries: list[list[str]]
) -> list[float]:
    """Answers to division queries."""
    # a / b = k  gives edges a -> b (k) and b -> a (1 / k)
    graph: dict[str, list[tuple[str, float]]] = defaultdict(list)
    for (a, b), k in zip(equations, values):
        graph[a].append((b, k))
        graph[b].append((a, 1 / k))

    def dfs(x: str, y: str, seen: set[str]) -> float:
        """Product of ratios along a path x -> y, or -1."""
        if x == y:
            return 1.0
        seen.add(x)
        for nb, w in graph[x]:
            if nb not in seen:
                found = dfs(nb, y, seen)
                if found > 0:
                    return w * found
        return -1.0

    # unknown variables have no answer
    return [dfs(x, y, set()) if x in graph and y in graph else -1.0 for x, y in queries]


res = calc_equation(
    [["a", "b"], ["b", "c"]],
    [2.0, 3.0],
    [["a", "c"], ["b", "a"], ["a", "e"], ["a", "a"], ["x", "x"]],
)
assert res == [6.0, 0.5, -1.0, 1.0, -1.0]
print(res)

[6.0, 0.5, -1.0, 1.0, -1.0]


## 2.2 BFS over state spaces

When nodes are *states* (words, lock combinations, board squares), generate neighbours on
the fly instead of building the graph. Ask: what is a state, what is one move, and how do
I keep the neighbour generation cheap?

### 2.2.1 [127. Word Ladder](https://leetcode.com/problems/word-ladder/) — ★★ · Hard

**Problem.** Fewest words in a transformation sequence from `begin` to `end`, changing one letter at a time, each intermediate word in the list.  
**Signal.** Shortest path in an implicit graph → BFS; neighbours via wildcard patterns.  
*New here: Wildcard buckets to find neighbours fast.*

**Fine print.** Ask before coding:
- Does the count include `begin`? → Yes, it is the number of words in the sequence.
- Is `begin` in the list? → Not necessarily.
- Same length, lowercase only? → Yes.

> 💡 **Key insight.** Two words are neighbours iff they share a pattern like `h*t`, so grouping words by pattern finds all neighbours without comparing every pair.

**Brute force →** Compare every pair of words to build edges → O(N² · L). Patterns → O(N · L²).

**Steps**
1. Map each pattern (one letter replaced by `*`) to its words.
2. BFS from `begin` with step count 1.
3. Neighbours = words sharing any of the current word's patterns; skip seen ones.
4. Empty each used bucket (its words are all queued); return the steps at `end`.

**Complexity**

- **Time:** O(N · L²) — N words × L patterns × O(L) to build each.
- **Space:** O(N · L²).

⚠️ **Trap.** If `end` is not in the list, the answer is 0 immediately.

In [23]:
def ladder_length(begin_word: str, end_word: str, word_list: list[str]) -> int:
    """Length of the shortest transformation sequence, or 0."""
    if end_word not in word_list:
        return 0
    # "h*t" -> every word matching it
    buckets: dict[str, list[str]] = defaultdict(list)
    for w in word_list:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1 :]].append(w)
    queue, seen = deque([(begin_word, 1)]), {begin_word}
    while queue:
        word, steps = queue.popleft()
        if word == end_word:
            return steps
        for i in range(len(word)):
            pattern = word[:i] + "*" + word[i + 1 :]
            for nb in buckets[pattern]:
                if nb not in seen:
                    seen.add(nb)
                    queue.append((nb, steps + 1))
            # every word in this bucket is queued; never scan it again
            buckets[pattern] = []
    return 0


words = ["hot", "dot", "dog", "lot", "log", "cog"]
assert ladder_length("hit", "cog", words) == 5
assert ladder_length("hit", "cog", words[:-1]) == 0
print(ladder_length("hit", "cog", words))

5


### 2.2.2 [433. Minimum Genetic Mutation](https://leetcode.com/problems/minimum-genetic-mutation/) — ★★ · Medium

**Problem.** Fewest single-character mutations (letters A/C/G/T) from `start` to `end`, each step in `bank`.  
**Signal.** Word Ladder with a 4-letter alphabet.  
*New here: Generating neighbours by trying every single-letter change.*

**Fine print.** Ask before coding:
- Is the start in the bank? → Not necessarily.
- End not in the bank? → Return −1.

> 💡 **Key insight.** With a 4-letter alphabet and length 8, each state has only 24 possible mutations, so trying them all is cheap.

**Brute force →** Compare against every bank string per state → O(|bank| · 8) per state, similar. Generating mutations keeps it simple.

**Steps**
1. Put the bank in a set.
2. BFS from `start`.
3. For each position and letter, form the mutation; queue it if it is in the bank and unseen.
4. Return the step count at `end`, or −1.

**Complexity**

- **Time:** O(|bank| · 8 · 4).
- **Space:** O(|bank|).

In [24]:
def min_mutation(start: str, end: str, bank: list[str]) -> int:
    """Fewest mutations, or -1."""
    valid = set(bank)
    queue, seen = deque([(start, 0)]), {start}
    while queue:
        gene, steps = queue.popleft()
        if gene == end:
            return steps
        # every single-letter change is a candidate neighbour
        for i in range(len(gene)):
            for ch in "ACGT":
                nb = gene[:i] + ch + gene[i + 1 :]
                if nb in valid and nb not in seen:
                    seen.add(nb)
                    queue.append((nb, steps + 1))
    return -1


assert min_mutation("AACCGGTT", "AACCGGTA", ["AACCGGTA"]) == 1
assert min_mutation("AACCGGTT", "AAACGGTA", ["AACCGGTA", "AACCGCTA", "AAACGGTA"]) == 2
print("ok")

ok


### 2.2.3 [752. Open the Lock](https://leetcode.com/problems/open-the-lock/) — ★ · Medium

**Problem.** 4-wheel lock from `"0000"`; each move turns one wheel by ±1; avoid `deadends`. Fewest moves to `target`?  
**Signal.** BFS over 10⁴ states.  
*New here: Forbidden states handled by pre-marking them visited.*

**Fine print.** Ask before coding:
- Is `"0000"` a dead end? → It can be; then return −1.
- Do wheels wrap around (9 → 0)? → Yes.

> 💡 **Key insight.** Deadends are just states you may never enter, so seeding them into `seen` removes them from the search.

**Brute force →** DFS over all move sequences → exponential. BFS over the 10⁴ states → O(10⁴ · 8).

**Steps**
1. `seen` starts with the deadends; if `"0000"` is one, return −1.
2. BFS from `"0000"`.
3. Each state has 8 neighbours (each wheel ±1, wrapping with `% 10`).
4. Return the steps when `target` is popped.

**Complexity**

- **Time:** O(10⁴ · 8).
- **Space:** O(10⁴).

In [25]:
def open_lock(deadends: list[str], target: str) -> int:
    """Fewest turns to reach target, or -1."""
    # deadends behave like already-visited states
    seen = set(deadends)
    if "0000" in seen:
        return -1
    seen.add("0000")
    queue = deque([("0000", 0)])
    while queue:
        state, steps = queue.popleft()
        if state == target:
            return steps
        # turn each wheel one step up or down
        for i in range(4):
            d = int(state[i])
            for nd in ((d + 1) % 10, (d - 1) % 10):
                nb = state[:i] + str(nd) + state[i + 1 :]
                if nb not in seen:
                    seen.add(nb)
                    queue.append((nb, steps + 1))
    return -1


assert open_lock(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
assert open_lock(["8888"], "0009") == 1
assert (
    open_lock(["8887", "8889", "8878", "8898", "8788", "8988", "7888", "9888"], "8888")
    == -1
)
print("ok")

ok


### 2.2.4 [909. Snakes and Ladders](https://leetcode.com/problems/snakes-and-ladders/) — ★★ · Medium

**Problem.** Boustrophedon-numbered board; roll 1–6 each move, snakes/ladders teleport. Fewest moves to the last square?  
**Signal.** BFS over squares; the tricky part is converting square number → (row, col).  
*New here: A coordinate mapping as the hard part of an easy BFS.*

**Fine print.** Ask before coding:
- Numbering starts at the bottom-left and alternates direction each row? → Yes.
- Do you follow at most one snake or ladder per move? → Yes, no chaining.

> 💡 **Key insight.** Square `s` is in row `n − 1 − (s − 1) // n` from the top, and the column direction flips every row; after that it is plain BFS on square numbers.

**Brute force →** DP over squares fails because snakes go backwards; BFS handles it.

**Steps**
1. `cell(s)` converts a square number to the board value (row from the bottom, alternating direction).
2. BFS from square 1 with 0 moves.
3. For each roll 1–6, follow a snake/ladder if present.
4. Return the moves when square `n²` is popped.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n²).

In [26]:
def snakes_and_ladders(board: list[list[int]]) -> int:
    """Fewest dice rolls to reach the final square."""
    n = len(board)

    def cell(s: int) -> int:
        """Board value at square s (1-based, boustrophedon from the bottom)."""
        q, r = divmod(s - 1, n)
        row = n - 1 - q
        # even rows from the bottom run left to right, odd rows right to left
        col = r if q % 2 == 0 else n - 1 - r
        return board[row][col]

    queue, seen = deque([(1, 0)]), {1}
    while queue:
        s, moves = queue.popleft()
        if s == n * n:
            return moves
        for nxt in range(s + 1, min(s + 6, n * n) + 1):
            # a snake or ladder moves you immediately
            dest = cell(nxt) if cell(nxt) != -1 else nxt
            if dest not in seen:
                seen.add(dest)
                queue.append((dest, moves + 1))
    return -1


b = [
    [-1, -1, -1, -1, -1, -1],
    [-1, -1, -1, -1, -1, -1],
    [-1, -1, -1, -1, -1, -1],
    [-1, 35, -1, -1, 13, -1],
    [-1, -1, -1, -1, -1, -1],
    [-1, 15, -1, -1, -1, -1],
]
assert snakes_and_ladders(b) == 4
assert snakes_and_ladders([[-1, -1], [-1, 3]]) == 1
print("ok")

ok


### 2.2.5 [815. Bus Routes](https://leetcode.com/problems/bus-routes/) — ★ · Hard

**Problem.** `routes[i]` is a bus loop over stops. Fewest buses to get from `source` to `target`?  
**Signal.** BFS where nodes are *buses*, not stops (one step = one bus ride).  
*New here: Choosing the right node type so that one BFS step = one unit of cost.*

**Fine print.** Ask before coding:
- Count buses, not stops? → Buses.
- `source == target`? → 0.

> 💡 **Key insight.** The cost is the number of buses, not stops, so make buses the BFS nodes: two buses are adjacent if they share a stop.

**Brute force →** BFS over stops with edges between every pair of stops on a route → O(Σ route²) edges. Buses as nodes with a stop → buses map → O(total stops).

**Steps**
1. Map stop → buses through it.
2. Start the BFS with every bus through `source` (1 ride).
3. Riding a bus visits its stops; reaching `target` returns the rides.
4. At each new stop, queue unused buses with rides + 1; mark stops and buses visited.

**Complexity**

- **Time:** O(total stops) — each bus and stop is processed once.
- **Space:** O(total stops).

In [27]:
def num_buses_to_destination(routes: list[list[int]], source: int, target: int) -> int:
    """Fewest buses from source to target, or -1."""
    if source == target:
        return 0
    stop_to_buses: dict[int, list[int]] = defaultdict(list)
    for bus, route in enumerate(routes):
        for stop in route:
            stop_to_buses[stop].append(bus)
    # BFS nodes are buses; every bus at the source costs one ride
    queue = deque((bus, 1) for bus in stop_to_buses[source])
    used_bus = set(stop_to_buses[source])
    seen_stop = {source}
    while queue:
        bus, rides = queue.popleft()
        # ride this bus to each of its stops
        for stop in routes[bus]:
            if stop == target:
                return rides
            if stop in seen_stop:
                continue
            seen_stop.add(stop)
            # transfer to any bus not taken yet
            for nb in stop_to_buses[stop]:
                if nb not in used_bus:
                    used_bus.add(nb)
                    queue.append((nb, rides + 1))
    return -1


assert num_buses_to_destination([[1, 2, 7], [3, 6, 7]], 1, 6) == 2
routes = [[7, 12], [4, 5, 15], [6], [15, 19], [9, 12, 13]]
assert num_buses_to_destination(routes, 15, 12) == -1
print("ok")

ok


---
# 3) Topological Sort

**In plain English.** Topological sort lines up tasks so that every task comes after
everything it depends on. Repeatedly take a task with no remaining prerequisites, do it,
and cross it off the requirements of the tasks waiting on it.

> **Analogy.** Getting dressed: socks before shoes, shirt before jacket. You can always
> start with anything that needs nothing first; if two items each require the other,
> you can never finish — that is a cycle.

**Kahn's algorithm** repeatedly removes nodes with in-degree 0. The removal order is a valid
ordering; nodes never removed sit on (or behind) a cycle.

```python
indeg = [0] * n; graph = defaultdict(list)
for a, b in edges: graph[a].append(b); indeg[b] += 1
queue = deque(i for i in range(n) if indeg[i] == 0)
order = []
while queue:
    u = queue.popleft(); order.append(u)
    for v in graph[u]:
        indeg[v] -= 1
        if indeg[v] == 0: queue.append(v)
# len(order) < n  ⇒ cycle
```

**Recognition signals**

- "prerequisites", "must be done before", "dependencies", "build order"
- "is it possible to finish" (= no cycle in a directed graph)
- "order of letters in an alien language"
- "peel leaves", "nodes that always terminate" (Kahn on a reversed or undirected graph)

**Common bugs**

- Reversing the edge direction: `[a, b]` = "b before a" means edge `b → a`.
- Forgetting nodes with no edges (they start with in-degree 0 and must be included).
- Alien Dictionary: a longer word before its own prefix (`"abc"` before `"ab"`) is invalid.
- Adding the same edge twice (inflates in-degree) — use a set of edges.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Undirected cycle check | use union–find or parent-tracking DFS; Kahn is for *directed* graphs |
| BFS shortest path | the queue holds "ready" nodes, not nodes at distance d |
| DFS post-order topo sort | equivalent result; Kahn makes cycle detection a simple count |

**Learning path.** Cycle check (Course Schedule) → produce the order → derive the edges
yourself (Alien Dictionary) → peel leaves of an undirected tree (Minimum Height Trees) →
Kahn on the reversed graph (Eventual Safe States).

## 3.1 Kahn's algorithm and DFS ordering

Every question reuses one helper, `topo_order`. The work is in building the right edges:
which direction, from what data, and on which graph (original, reversed, undirected).

### 3.1.1 [207. Course Schedule](https://leetcode.com/problems/course-schedule/) — ★★★ · Medium

**Problem.** `prerequisites[i] = [a, b]` means take `b` before `a`. Can all courses be finished?  
**Signal.** Prerequisites → directed graph; feasible ⇔ no cycle ⇔ topological order covers all.  
*New here: Kahn's algorithm and "order shorter than n ⇒ cycle".*

**Fine print.** Ask before coding:
- Can there be duplicate prerequisite pairs? → No.
- Is a self-dependency a cycle? → Yes.

> 💡 **Key insight.** Courses on a cycle never reach in-degree 0, so all courses can be taken iff Kahn's algorithm removes every node.

**Brute force →** DFS cycle detection with three colours → also O(V + E). Kahn is iterative and easier to get right.

**Steps**
1. Edge `b → a` for each prerequisite `[a, b]`; count in-degrees.
2. Queue every course with in-degree 0.
3. Pop a course, append it, decrement its successors; queue those reaching 0.
4. Feasible iff the order contains all courses.

**Complexity**

- **Time:** O(V + E).
- **Space:** O(V + E).

In [28]:
def topo_order(n: int, edges: list[tuple[int, int]]) -> list[int]:
    """Kahn's order for edges u -> v (shorter than n if cyclic)."""
    graph = defaultdict(list)
    indeg = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indeg[v] += 1
    # nodes with nothing left to wait for
    queue = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        # u is done: its dependants lose one requirement
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    return order


def can_finish(num_courses: int, prerequisites: list[list[int]]) -> bool:
    """True if the prerequisite graph is acyclic."""
    # [a, b] means b must come before a
    edges = [(b, a) for a, b in prerequisites]
    return len(topo_order(num_courses, edges)) == num_courses


assert can_finish(2, [[1, 0]])
assert not can_finish(2, [[1, 0], [0, 1]])
print("ok")

ok


### 3.1.2 [210. Course Schedule II](https://leetcode.com/problems/course-schedule-ii/) — ★★ · Medium

**Problem.** Return any valid order to take all courses, or `[]` if impossible.  
**Signal.** Same as Course Schedule, return the order itself.  
*New here: Returning the order, with the cycle check as a guard.*

**Fine print.** Ask before coding:
- Any valid order? → Yes.
- Impossible? → Return `[]`.

> 💡 **Key insight.** Kahn's removal order is itself a valid schedule whenever it contains every course.

**Brute force →** Try permutations and check constraints → factorial. Kahn → O(V + E).

**Steps**
1. Build edges `b → a`.
2. Run Kahn's algorithm.
3. Return the order if it has all courses, else `[]`.

**Complexity**

- **Time:** O(V + E).
- **Space:** O(V + E).

In [29]:
def find_order(num_courses: int, prerequisites: list[list[int]]) -> list[int]:
    """A valid course order, or empty."""
    order = topo_order(num_courses, [(b, a) for a, b in prerequisites])
    # a short order means some courses sit on a cycle
    return order if len(order) == num_courses else []


assert find_order(2, [[1, 0]]) == [0, 1]
order = find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]])
assert order[0] == 0 and order[-1] == 3
assert find_order(2, [[0, 1], [1, 0]]) == []
print(order)

[0, 1, 2, 3]


### 3.1.3 [269. Alien Dictionary](https://leetcode.com/problems/alien-dictionary/) — ★★★ · Hard · Premium

**Problem.** Words sorted in an unknown alphabet; return a valid letter order (or `""` if invalid).  
**Signal.** Adjacent words give ordering constraints → topological sort on letters.  
*New here: Extracting the edges yourself from indirect evidence.*

**Fine print.** Ask before coding:
- A prefix that comes after the longer word (`"abc"` before `"ab"`)? → Invalid, return `""`.
- Several valid orders? → Return any.
- Include letters that have no order constraints? → Yes.

> 💡 **Key insight.** Two adjacent sorted words differ first at some position, and only that first difference tells you something: letter `a` comes before letter `b`.

**Brute force →** Try every letter permutation and check the word order → 26! — impossible. Edges + topological sort → O(total characters).

**Steps**
1. Collect all letters (each is a node, even without edges).
2. For each adjacent pair, the first differing letters give an edge `a → b`.
3. No difference and the first word is longer → invalid, return `""`.
4. Topologically sort; a cycle (short order) → `""`.

**Complexity**

- **Time:** O(total characters).
- **Space:** O(alphabet + edges).

In [30]:
def alien_order(words: list[str]) -> str:
    """Letter order consistent with the sorted words."""
    letters = sorted({ch for w in words for ch in w})
    index = {ch: i for i, ch in enumerate(letters)}
    edges = set()
    for w1, w2 in zip(words, words[1:]):
        # only the first differing position carries information
        for a, b in zip(w1, w2):
            if a != b:
                edges.add((index[a], index[b]))
                break
        # no difference: a longer word may not come before its prefix
        else:
            if len(w1) > len(w2):
                return ""
    order = topo_order(len(letters), sorted(edges))
    return "".join(letters[i] for i in order) if len(order) == len(letters) else ""


assert alien_order(["wrt", "wrf", "er", "ett", "rftt"]) == "wertf"
assert alien_order(["z", "x"]) == "zx"
assert alien_order(["z", "x", "z"]) == ""
assert alien_order(["abc", "ab"]) == ""
print(alien_order(["wrt", "wrf", "er", "ett", "rftt"]))

wertf


### 3.1.4 [310. Minimum Height Trees](https://leetcode.com/problems/minimum-height-trees/) — ★★ · Medium

**Problem.** Roots of all minimum-height trees of an undirected tree.  
**Signal.** Peel leaves layer by layer (topological sort on an undirected tree); the last 1–2 nodes are the centres.  
*New here: Kahn-style peeling on an undirected graph (degree 1 instead of in-degree 0).*

**Fine print.** Ask before coding:
- Is the input a valid tree? → Yes.
- Can there be one or two answers? → Yes, at most two.
- `n == 1`? → `[0]`.

> 💡 **Key insight.** The best root is the centre of the longest path; peeling all leaves at once, layer by layer, shrinks every path from both ends until only the 1 or 2 centre nodes remain.

**Brute force →** BFS height from every node → O(n²). Leaf peeling → O(n).

**Steps**
1. Build adjacency sets; start with all leaves (degree 1).
2. While more than 2 nodes remain, remove the current leaves.
3. Each removal lowers a neighbour's degree; neighbours reaching degree 1 are the next leaves.
4. The final leaves are the answer.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [31]:
def find_min_height_trees(n: int, edges: list[list[int]]) -> list[int]:
    """Centres of the tree."""
    if n == 1:
        return [0]
    graph = defaultdict(set)
    for a, b in edges:
        graph[a].add(b)
        graph[b].add(a)
    leaves = [i for i in range(n) if len(graph[i]) == 1]
    remaining = n
    # peel one full layer of leaves at a time
    while remaining > 2:
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            nb = graph[leaf].pop()
            graph[nb].discard(leaf)
            # the neighbour just became a leaf of the next layer
            if len(graph[nb]) == 1:
                nxt.append(nb)
        leaves = nxt
    return sorted(leaves)


assert find_min_height_trees(4, [[1, 0], [1, 2], [1, 3]]) == [1]
assert find_min_height_trees(6, [[3, 0], [3, 1], [3, 2], [3, 4], [5, 4]]) == [3, 4]
print("ok")

ok


### 3.1.5 [802. Find Eventual Safe States](https://leetcode.com/problems/find-eventual-safe-states/) — ★ · Medium

**Problem.** A node is safe if every path from it ends at a terminal node. Return all safe nodes.  
**Signal.** Reverse the edges and run Kahn from terminal nodes (out-degree 0).  
*New here: Kahn on the *reversed* graph.*

**Fine print.** Ask before coding:
- Return in sorted order? → Yes.
- Is a node on a cycle unsafe? → Yes, and so is any node that can reach a cycle.

> 💡 **Key insight.** A node is safe iff all its outgoing edges lead to safe nodes; reversing edges turns "out-degree 0 so far" into "in-degree 0", which Kahn peels naturally.

**Brute force →** DFS from every node checking for cycles → O(V · (V + E)). Three-colour DFS → O(V + E). Reversed Kahn → O(V + E), no recursion.

**Steps**
1. Reverse every edge `u → v` to `v → u`.
2. Terminal nodes now have in-degree 0.
3. Run Kahn; every node removed is safe.
4. Return them sorted.

**Complexity**

- **Time:** O(V + E).
- **Space:** O(V + E).

In [32]:
def eventual_safe_nodes(graph: list[list[int]]) -> list[int]:
    """Nodes from which every path terminates."""
    # reversed edges: a node is freed once all its successors are safe
    reversed_edges = [(v, u) for u, outs in enumerate(graph) for v in outs]
    return sorted(topo_order(len(graph), reversed_edges))


assert eventual_safe_nodes([[1, 2], [2, 3], [5], [0], [5], [], []]) == [2, 4, 5, 6]
assert eventual_safe_nodes([[1, 2, 3, 4], [1, 2], [3, 4], [0, 4], []]) == [4]
print("ok")

ok


---
# 4) Union-Find

**In plain English.** Union–find keeps track of which items are in the same group while
groups keep merging. Each group has one representative (the root); two items are connected
iff they have the same root.

> **Analogy.** Companies merging: each employee can ask "who is my ultimate parent
> company?". When two companies merge, one CEO starts reporting to the other — nobody else
> has to move.

With **path compression** (point nodes closer to the root while searching) and **union by
size** (attach the smaller tree under the larger), each operation is effectively O(1).

**Recognition signals**

- edges arrive one at a time and you ask "are these already connected?"
- "redundant connection", "cycle in an undirected graph"
- "merge accounts / groups that share something"
- equalities and inequalities between variables
- "number of components" while adding edges

**Common bugs**

- Comparing `parent[a] == parent[b]` instead of `find(a) == find(b)`.
- Forgetting to map non-integer items (emails, letters) to indices first.
- Processing `!=` constraints before all `==` unions are done.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| DFS / BFS components | recompute from scratch on a fixed graph; union–find updates incrementally |
| Topological sort | for *directed* dependencies; union–find ignores direction |
| Kruskal's MST | union–find is the cycle check inside Kruskal |

**Learning path.** Detect the cycle-closing edge → group by shared elements (accounts) →
equalities then inequalities → count components and spare edges.

In [33]:
class DSU:
    """Disjoint set union with path compression and union by size."""

    def __init__(self, n: int) -> None:
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> bool:
        """Merge groups; False if already joined."""
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


d = DSU(4)
print(d.union(0, 1), d.union(1, 0), d.find(1) == d.find(0))

True False True


## 4.1 Dynamic connectivity

The DSU above is the whole toolkit: `union` returns `False` when two items were already
connected — that single bit answers most questions.

### 4.1.1 [684. Redundant Connection](https://leetcode.com/problems/redundant-connection/) — ★★ · Medium

**Problem.** A tree plus one extra edge; return the edge that can be removed (the last such in input).  
**Signal.** The first edge joining two already-connected nodes closes the cycle → union–find.  
*New here: Using `union`'s return value as a cycle detector.*

**Fine print.** Ask before coding:
- Several candidate edges? → Return the one that appears last in the input.
- Undirected? → Yes (684; the directed version is 685).

> 💡 **Key insight.** Adding edges one by one, the edge whose endpoints are already connected is exactly the one that closes the cycle.

**Brute force →** Remove each edge and test whether the rest is a tree → O(n²). Union–find → O(n α(n)).

**Steps**
1. Create a DSU over nodes 1..n.
2. Union each edge in order.
3. The first union that returns `False` is the answer.

**Complexity**

- **Time:** O(n α(n)).
- **Space:** O(n).

In [34]:
def find_redundant_connection(edges: list[list[int]]) -> list[int]:
    """Edge that creates the cycle."""
    dsu = DSU(len(edges) + 1)
    for a, b in edges:
        # already connected: this edge closes a cycle
        if not dsu.union(a, b):
            return [a, b]
    return []


assert find_redundant_connection([[1, 2], [1, 3], [2, 3]]) == [2, 3]
assert find_redundant_connection([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]) == [1, 4]
print("ok")

ok


### 4.1.2 [721. Accounts Merge](https://leetcode.com/problems/accounts-merge/) — ★★ · Medium

**Problem.** Merge accounts that share any email; output name + sorted emails.  
**Signal.** "Share any element → same group" → union–find on account indices.  
*New here: Unioning items through a shared key (email → first owner).*

**Fine print.** Ask before coding:
- Can two different people share a name? → Yes, so merge by email, not by name.
- Output emails sorted? → Yes. Accounts in any order? → Yes.

> 💡 **Key insight.** Two accounts belong together iff some email links them, so union each account with the first account that owned each of its emails.

**Brute force →** Compare every pair of accounts for common emails → O(A² · E). Email → owner map + union–find → O(N α(N) + N log N).

**Steps**
1. For each email, remember its first owner; union later owners with it.
2. Group emails by `find(owner)`.
3. Output each group's name (from the root account) plus sorted emails.

**Complexity**

- **Time:** O(N α(N) + N log N) for N emails (sorting dominates).
- **Space:** O(N).

In [35]:
def accounts_merge(accounts: list[list[str]]) -> list[list[str]]:
    """Merged accounts with sorted emails."""
    dsu = DSU(len(accounts))
    # email -> first account that listed it
    owner: dict[str, int] = {}
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            # shared email: the two accounts are the same person
            if e in owner:
                dsu.union(i, owner[e])
            else:
                owner[e] = i
    # collect emails under each group's root account
    groups: dict[int, list[str]] = defaultdict(list)
    for e, i in owner.items():
        groups[dsu.find(i)].append(e)
    return [[accounts[root][0]] + sorted(es) for root, es in groups.items()]


accounts = [
    ["John", "johnsmith@mail.com", "john_newyork@mail.com"],
    ["John", "johnsmith@mail.com", "john00@mail.com"],
    ["Mary", "mary@mail.com"],
    ["John", "johnnybravo@mail.com"],
]
out = sorted(accounts_merge(accounts))
assert out == [
    ["John", "john00@mail.com", "john_newyork@mail.com", "johnsmith@mail.com"],
    ["John", "johnnybravo@mail.com"],
    ["Mary", "mary@mail.com"],
]
print(out)

[['John', 'john00@mail.com', 'john_newyork@mail.com', 'johnsmith@mail.com'], ['John', 'johnnybravo@mail.com'], ['Mary', 'mary@mail.com']]


### 4.1.3 [990. Satisfiability of Equality Equations](https://leetcode.com/problems/satisfiability-of-equality-equations/) — ★ · Medium

**Problem.** Equations `a==b` / `a!=b` over letters; can all hold simultaneously?  
**Signal.** Equalities define groups (union), inequalities must cross groups.  
*New here: Two phases: build groups first, then validate constraints.*

**Fine print.** Ask before coding:
- Variables are lowercase letters only? → Yes, 26 nodes.
- Is `a!=a` a contradiction? → Yes.

> 💡 **Key insight.** Equalities are transitive, so they form groups; an inequality is violated iff both sides land in the same group.

**Brute force →** Assign values with backtracking → exponential. Union then check → O(n α(26)).

**Steps**
1. Union the two letters of every `==`.
2. For every `!=`, the letters must have different roots.
3. Return whether all checks pass.

**Complexity**

- **Time:** O(n α(26)).
- **Space:** O(26).

⚠️ **Trap.** Checking `!=` before all `==` are processed misses later merges.

In [36]:
def equations_possible(equations: list[str]) -> bool:
    """True if all equations can be satisfied."""
    dsu = DSU(26)
    # all equalities first: they define the groups
    for eq in equations:
        if eq[1] == "=":
            dsu.union(ord(eq[0]) - 97, ord(eq[3]) - 97)
    # every inequality must cross two different groups
    return all(
        dsu.find(ord(eq[0]) - 97) != dsu.find(ord(eq[3]) - 97)
        for eq in equations
        if eq[1] == "!"
    )


assert not equations_possible(["a==b", "b!=a"])
assert equations_possible(["b==a", "a==b"])
assert not equations_possible(["a==b", "b==c", "a!=c"])
print("ok")

ok


### 4.1.4 [1319. Number of Operations to Make Network Connected](https://leetcode.com/problems/number-of-operations-to-make-network-connected/) — ★ · Medium

**Problem.** Move cables (edges) to connect all `n` computers; fewest moves, or −1.  
**Signal.** Count components and spare (redundant) cables.  
*New here: Counting components while edges are added.*

**Fine print.** Ask before coding:
- Not enough cables (`edges < n − 1`)? → Return −1.
- Can any redundant cable be moved? → Yes.

> 💡 **Key insight.** With at least `n − 1` cables there are always enough spares, and joining `c` components needs exactly `c − 1` moves.

**Brute force →** Simulate moving cables → unnecessary. Count components with union–find → O(E α(n)).

**Steps**
1. Fewer than `n − 1` cables → −1.
2. Start with `n` components; each successful union removes one.
3. Return `components − 1`.

**Complexity**

- **Time:** O(E α(n)).
- **Space:** O(n).

In [37]:
def make_connected(n: int, connections: list[list[int]]) -> int:
    """Cable moves needed to connect all computers."""
    # a connected network of n nodes needs n - 1 cables
    if len(connections) < n - 1:
        return -1
    dsu = DSU(n)
    components = n
    for a, b in connections:
        # a successful union merges two components
        components -= dsu.union(a, b)
    return components - 1


assert make_connected(4, [[0, 1], [0, 2], [1, 2]]) == 1
assert make_connected(6, [[0, 1], [0, 2], [0, 3], [1, 2], [1, 3]]) == 2
assert make_connected(6, [[0, 1], [0, 2], [0, 3], [1, 2]]) == -1
print("ok")

ok


---
# 5) Weighted Shortest Paths and MST

**In plain English.** When edges have different costs, BFS's "fewest edges" is no longer
"cheapest". Dijkstra always expands the cheapest unfinished node next; with non-negative
costs, that node's cost can never improve later, so it is final.

> **Analogy.** Water poured at the source spreads along pipes at constant speed; it
> reaches each junction first along the cheapest path. Dijkstra simulates that flood
> with a priority queue.

```python
dist = {src: 0}; heap = [(0, src)]
while heap:
    d, u = heapq.heappop(heap)
    if d > dist[u]: continue           # stale entry
    for v, w in graph[u]:
        if d + w < dist.get(v, inf):
            dist[v] = d + w; heapq.heappush(heap, (d + w, v))
```

The same loop with a different "cost combination" (max instead of sum, product instead of
sum) solves bottleneck and probability variants. A **minimum spanning tree** connects all
nodes at minimum total cost — Prim's algorithm is Dijkstra's loop with "edge weight"
instead of "path length".

**Recognition signals**

- "weighted", "cost", "time", "price", "effort", "probability" on edges
- "cheapest / fastest / minimum effort from A to B"
- "at most k stops" → Bellman–Ford with k + 1 rounds
- "connect all points with minimum total cost" → MST
- "use every edge exactly once" → Eulerian path

**Common bugs**

- Not skipping stale heap entries (`if d > dist[u]: continue`) → slow, sometimes wrong.
- Using Dijkstra with negative weights.
- Dijkstra for "at most k stops": a cheaper path with too many stops blocks the valid one.
- Bellman–Ford without copying the previous round → paths with too many edges sneak in.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| BFS | all edges cost 1; Dijkstra handles different weights |
| Bellman–Ford | handles an edge-count limit and negative weights; O(k · E) |
| MST (Prim/Kruskal) | minimises the *total* of chosen edges, not a path from a source |
| Binary search + BFS | alternative for bottleneck problems ("max step ≤ X feasible?") |

**Learning path.** Plain Dijkstra (Network Delay) → bottleneck variants (Minimum Effort,
Swim in Rising Water) → product with max-heap (Maximum Probability) → edge-count limit with
Bellman–Ford (Cheapest Flights) → MST with Prim (Connect All Points) → Eulerian path
(Reconstruct Itinerary).

## 5.1 Dijkstra and variants

Dijkstra's loop stays the same; what changes is how a path's cost combines (`+`, `max`,
`×`) and when you can stop (first time the target is popped).

### 5.1.1 [743. Network Delay Time](https://leetcode.com/problems/network-delay-time/) — ★★ · Medium

**Problem.** Directed weighted edges; time for a signal from `k` to reach all nodes (−1 if not all).  
**Signal.** Single-source shortest paths, non-negative weights → Dijkstra.  
*New here: The Dijkstra template with stale-entry skipping.*

**Fine print.** Ask before coding:
- Non-negative weights? → Yes, so Dijkstra applies.
- 1-indexed nodes? → Yes.
- Some node unreachable? → −1.

> 💡 **Key insight.** The signal reaches every node along its shortest path, so the answer is the largest shortest-path distance.

**Brute force →** Bellman–Ford → O(V · E). Dijkstra with a heap → O(E log V).

**Steps**
1. Build the adjacency list.
2. Dijkstra from `k`: pop the closest node, skip stale entries, relax its edges.
3. If not every node got a distance, return −1.
4. Otherwise return the maximum distance.

**Complexity**

- **Time:** O(E log V).
- **Space:** O(V + E).

In [38]:
def network_delay_time(times: list[list[int]], n: int, k: int) -> int:
    """Time for all nodes to receive the signal."""
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = {k: 0}
    heap = [(0, k)]
    while heap:
        # closest unfinished node: its distance is final
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        # relax outgoing edges
        for v, w in graph[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v] = d + w
                heapq.heappush(heap, (d + w, v))
    # the last node to hear the signal decides the time
    return max(dist.values()) if len(dist) == n else -1


assert network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
assert network_delay_time([[1, 2, 1]], 2, 2) == -1
print("ok")

ok


### 5.1.2 [1631. Path With Minimum Effort](https://leetcode.com/problems/path-with-minimum-effort/) — ★ · Medium

**Problem.** Grid of heights; a path's effort is its largest absolute step difference. Minimise effort from top-left to bottom-right.  
**Signal.** Bottleneck shortest path → Dijkstra where path cost = max edge, not sum.  
*New here: Dijkstra with `max` instead of `+`.*

**Fine print.** Ask before coding:
- 4-directional? → Yes.
- Is effort the maximum step along the path, not the sum? → Yes.

> 💡 **Key insight.** A path's effort never decreases as it gets longer (the max can only grow), which is all Dijkstra needs to be correct.

**Brute force →** Binary search on the effort + BFS feasibility → O(mn log H). Dijkstra on bottleneck cost → O(mn log mn).

**Steps**
1. Heap of `(effort, r, c)` starting at `(0, 0, 0)`.
2. Pop; if it is the target, return the effort.
3. Skip stale entries.
4. For each neighbour, new effort = `max(effort, |height difference|)`; push if it improves.

**Complexity**

- **Time:** O(mn log(mn)).
- **Space:** O(mn).

In [39]:
def minimum_effort_path(heights: list[list[int]]) -> int:
    """Least possible maximum step on a path."""
    rows, cols = len(heights), len(heights[0])
    best = {(0, 0): 0}
    heap = [(0, 0, 0)]
    while heap:
        d, r, c = heapq.heappop(heap)
        # first time the target is popped, its effort is optimal
        if (r, c) == (rows - 1, cols - 1):
            return d
        if d > best[(r, c)]:
            continue
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                # path cost is the largest single step so far
                nd = max(d, abs(heights[nr][nc] - heights[r][c]))
                if nd < best.get((nr, nc), float("inf")):
                    best[(nr, nc)] = nd
                    heapq.heappush(heap, (nd, nr, nc))
    return 0


assert minimum_effort_path([[1, 2, 2], [3, 8, 2], [5, 3, 5]]) == 2
assert minimum_effort_path([[1, 2, 3], [3, 8, 4], [5, 3, 5]]) == 1
print("ok")

ok


### 5.1.3 [778. Swim in Rising Water](https://leetcode.com/problems/swim-in-rising-water/) — ★★ · Hard

**Problem.** At time `t` you can swim through cells with elevation ≤ `t`. Earliest time to reach the bottom-right corner?  
**Signal.** Minimise the maximum cell on the path → bottleneck Dijkstra.  
*New here: Node-weighted bottleneck: the cost is the highest *cell*, not step.*

**Fine print.** Ask before coding:
- Are the elevations a permutation of `0..n²−1`? → Yes, all unique.
- Is swimming instant once cells are ≤ `t`? → Yes.

> 💡 **Key insight.** The earliest time is the smallest possible "highest cell on the path", which a heap ordered by that maximum finds greedily.

**Brute force →** Try every `t` with BFS → O(n² · max). Binary search on `t` + BFS → O(n² log n²). Bottleneck Dijkstra → O(n² log n).

**Steps**
1. Heap `(grid[0][0], 0, 0)`; mark visited on push.
2. Pop the lowest frontier cell; if it is the target, return its time.
3. Push each unvisited neighbour with `max(t, grid[nr][nc])`.

**Complexity**

- **Time:** O(n² log n).
- **Space:** O(n²).

In [40]:
def swim_in_water(grid: list[list[int]]) -> int:
    """Earliest time to reach the far corner."""
    n = len(grid)
    # (highest elevation on the path so far, row, col)
    heap = [(grid[0][0], 0, 0)]
    seen = {(0, 0)}
    while heap:
        t, r, c = heapq.heappop(heap)
        if (r, c) == (n - 1, n - 1):
            return t
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and (nr, nc) not in seen:
                seen.add((nr, nc))
                # you must wait until the water covers the higher cell
                heapq.heappush(heap, (max(t, grid[nr][nc]), nr, nc))
    return -1


assert swim_in_water([[0, 2], [1, 3]]) == 3
g = [
    [0, 1, 2, 3, 4],
    [24, 23, 22, 21, 5],
    [12, 13, 14, 15, 16],
    [11, 17, 18, 19, 20],
    [10, 9, 8, 7, 6],
]
assert swim_in_water(g) == 16
print("ok")

ok


### 5.1.4 [1514. Path with Maximum Probability](https://leetcode.com/problems/path-with-maximum-probability/) — ★ · Medium

**Problem.** Undirected edges with success probabilities; maximise the probability of reaching `end`.  
**Signal.** Dijkstra with products and a max-heap.  
*New here: Maximising instead of minimising (negate for the min-heap).*

**Fine print.** Ask before coding:
- Undirected? → Yes.
- No path? → 0.
- Probabilities in `[0, 1]`? → Yes, so multiplying never increases, and Dijkstra with max works.

> 💡 **Key insight.** Multiplying by probabilities ≤ 1 never increases a path's value, which mirrors "adding non-negative weights never decreases a cost" — so Dijkstra's greedy choice still holds.

**Brute force →** Bellman–Ford on products → O(V · E). Dijkstra → O(E log V).

**Steps**
1. Build the undirected graph.
2. `best[start] = 1`; heap of `(−probability, node)`.
3. Pop the most probable node; return if it is `end`; skip stale entries.
4. Relax neighbours with `p * w` when it beats their best.

**Complexity**

- **Time:** O(E log V).
- **Space:** O(V + E).

In [41]:
def max_probability(
    n: int, edges: list[list[int]], succ_prob: list[float], start: int, end: int
) -> float:
    """Highest success probability from start to end."""
    graph = defaultdict(list)
    for (a, b), p in zip(edges, succ_prob):
        graph[a].append((b, p))
        graph[b].append((a, p))
    best = [0.0] * n
    best[start] = 1.0
    # negated probabilities turn the min-heap into a max-heap
    heap = [(-1.0, start)]
    while heap:
        p, u = heapq.heappop(heap)
        p = -p
        if u == end:
            return p
        if p < best[u]:
            continue
        for v, w in graph[u]:
            # probabilities multiply along a path
            if p * w > best[v]:
                best[v] = p * w
                heapq.heappush(heap, (-best[v], v))
    return 0.0


assert max_probability(3, [[0, 1], [1, 2], [0, 2]], [0.5, 0.5, 0.2], 0, 2) == 0.25
assert max_probability(3, [[0, 1]], [0.5], 0, 2) == 0.0
print("ok")

ok


### 5.1.5 [787. Cheapest Flights Within K Stops](https://leetcode.com/problems/cheapest-flights-within-k-stops/) — ★★ · Medium

**Problem.** Cheapest price from `src` to `dst` using at most `k` stops (k + 1 flights).  
**Signal.** Edge-count limit → Bellman–Ford for `k + 1` rounds.  
*New here: Bellman–Ford rounds as "paths with at most i edges".*

**Fine print.** Ask before coding:
- Does `k` count stops, so `k + 1` flights? → Yes.
- Unreachable? → −1.
- Are prices positive? → Yes.

> 💡 **Key insight.** After round `i` of relaxing every edge from the *previous* round's prices, each price is the cheapest using at most `i` flights.

**Brute force →** DFS over all routes with ≤ k stops → exponential. Dijkstra on (cost, stops) works but is subtle. Bellman–Ford → O(k · E).

**Steps**
1. `prices[src] = 0`, everything else ∞.
2. Repeat `k + 1` times: copy prices, relax every flight using the old prices into the copy.
3. Replace prices with the copy.
4. Return `prices[dst]`, or −1 if still ∞.

**Complexity**

- **Time:** O(k · E).
- **Space:** O(n).

⚠️ **Trap.** Relaxing in place (without the copy) lets one round chain several flights.

In [42]:
def find_cheapest_price(
    n: int, flights: list[list[int]], src: int, dst: int, k: int
) -> int:
    """Cheapest cost with at most k stops, or -1."""
    inf = float("inf")
    prices = [inf] * n
    prices[src] = 0
    # round i allows paths with at most i flights
    for _ in range(k + 1):
        # read last round's prices, write into a copy
        nxt = prices.copy()
        for u, v, w in flights:
            if prices[u] + w < nxt[v]:
                nxt[v] = prices[u] + w
        prices = nxt
    return -1 if prices[dst] == inf else prices[dst]


flights = [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]]
assert find_cheapest_price(4, flights, 0, 3, 1) == 700
assert find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1) == 200
assert find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0) == 500
print("ok")

ok


### 5.1.6 [1584. Min Cost to Connect All Points](https://leetcode.com/problems/min-cost-to-connect-all-points/) — ★★ · Medium

**Problem.** Connect all points with minimum total Manhattan distance.  
**Signal.** Minimum spanning tree on a complete graph → Prim's algorithm.  
*New here: Minimum spanning tree with Prim's algorithm (array version for dense graphs).*

**Fine print.** Ask before coding:
- Manhattan distance? → Yes.
- Unique points? → Yes.
- Dense graph (`n ≤ 1000`)? → Yes, so O(n²) Prim beats sorting all n² edges.

> 💡 **Key insight.** The cheapest edge leaving the current tree is always safe to add (cut property), so grow the tree one closest point at a time.

**Brute force →** Kruskal on all n² edges → O(n² log n). Array-based Prim → O(n²), which is optimal for a complete graph.

**Steps**
1. `best[i]` = cheapest edge from the tree to point `i`; start with point 0 at cost 0.
2. Pick the outside point with the smallest `best`; add it and its cost.
3. Update `best` for all outside points using distances from the new point.
4. Repeat n times.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n).

In [43]:
def min_cost_connect_points(points: list[list[int]]) -> int:
    """MST weight under Manhattan distance."""
    n = len(points)
    # cheapest known edge from the tree to each point
    best = [float("inf")] * n
    in_tree = [False] * n
    best[0] = 0
    total = 0
    for _ in range(n):
        # closest point not yet in the tree
        u = min((i for i in range(n) if not in_tree[i]), key=best.__getitem__)
        in_tree[u] = True
        total += best[u]
        # the new point may offer cheaper edges to the rest
        for v in range(n):
            if not in_tree[v]:
                d = abs(points[u][0] - points[v][0]) + abs(points[u][1] - points[v][1])
                best[v] = min(best[v], d)
    return total


assert min_cost_connect_points([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]) == 20
assert min_cost_connect_points([[3, 12], [-2, 5], [-4, 1]]) == 18
print("ok")

ok


### 5.1.7 [332. Reconstruct Itinerary](https://leetcode.com/problems/reconstruct-itinerary/) — ★★ · Hard

**Problem.** Use every ticket once starting from `"JFK"`; return the lexicographically smallest itinerary.  
**Signal.** Use every edge exactly once → Eulerian path (Hierholzer's algorithm).  
*New here: Hierholzer's algorithm: build the route backwards from dead ends.*

**Fine print.** Ask before coding:
- Must every ticket be used exactly once? → Yes.
- Is a valid itinerary guaranteed? → Yes.
- Can there be duplicate tickets? → Yes.

> 💡 **Key insight.** Greedily follow the smallest ticket; when an airport has no tickets left, it must be the *end* of the remaining route, so append it to a reversed route.

**Brute force →** Backtracking over ticket orders → exponential in bad cases. Hierholzer → O(E log E).

**Steps**
1. Sort tickets so each airport's destinations pop in lexical order.
2. Stack starts at `"JFK"`.
3. While the top airport has tickets, fly the smallest one (push it).
4. When stuck, pop the airport onto the route; reverse the route at the end.

**Complexity**

- **Time:** O(E log E) — sorting the tickets.
- **Space:** O(E).

In [44]:
def find_itinerary(tickets: list[list[str]]) -> list[str]:
    """Smallest-lexical itinerary using every ticket."""
    # reverse-sorted lists so pop() yields the smallest destination
    graph = defaultdict(list)
    for a, b in sorted(tickets, reverse=True):
        graph[a].append(b)
    route: list[str] = []
    stack = ["JFK"]
    while stack:
        # keep flying while tickets remain
        while graph[stack[-1]]:
            stack.append(graph[stack[-1]].pop())
        # stuck airport: it closes the remaining route
        route.append(stack.pop())
    return route[::-1]


assert find_itinerary(
    [["MUC", "LHR"], ["JFK", "MUC"], ["SFO", "SJC"], ["LHR", "SFO"]]
) == [
    "JFK",
    "MUC",
    "LHR",
    "SFO",
    "SJC",
]
tickets = [
    ["JFK", "SFO"],
    ["JFK", "ATL"],
    ["SFO", "ATL"],
    ["ATL", "JFK"],
    ["ATL", "SFO"],
]
assert find_itinerary(tickets) == ["JFK", "ATL", "JFK", "SFO", "ATL", "SFO"]
assert find_itinerary([["JFK", "KUL"], ["JFK", "NRT"], ["NRT", "JFK"]]) == [
    "JFK",
    "NRT",
    "JFK",
    "KUL",
]
print("ok")

ok


---
# 6) Backtracking

**In plain English.** Backtracking builds answers one choice at a time. After each choice
you explore everything that follows, then *undo* the choice and try the next one. It is a
DFS over a tree of decisions; pruning cuts off branches that can no longer succeed.

> **Analogy.** Walking a hedge maze with a ball of string: at every fork take a path; at a
> dead end, wind the string back to the last fork and try the next path.

```python
def backtrack(start, path):
    if is_complete(path):
        out.append(path.copy()); return
    for i in range(start, len(choices)):
        if not valid(choices[i]): continue          # prune
        path.append(choices[i])                     # choose
        backtrack(i + 1, path)                      # explore (i for reuse, 0 for permutations)
        path.pop()                                  # un-choose
```

| Problem type | Recurse with | Avoid duplicates by |
|---|---|---|
| subsets / combinations | `i + 1` | sorting + skip `nums[i] == nums[i-1]` when `i > start` |
| unlimited reuse | `i` | — |
| permutations | loop from 0 with a `used` array | sorting + skip if the previous equal value is unused |

**Recognition signals**

- "all subsets / combinations / permutations / partitions"
- "generate all valid …" (parentheses, IP addresses, boards)
- small input (n ≤ 15–20), output can be exponential
- constraint puzzles: N-Queens, Sudoku, fill k buckets
- a path in a grid that may not reuse cells (Word Search)

**Common bugs**

- `out.append(path)` instead of `path.copy()` — every saved answer is the same list.
- Forgetting to undo state (pop, unmark the cell, remove from the set).
- Duplicate results from equal values — sort first, then skip equal siblings.
- `i` vs `i + 1` in the recursive call (reuse allowed or not).
- No pruning → TLE; sort so you can `break` once a choice is too big.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| DFS on a graph | visits each node once; backtracking revisits states through different paths |
| Dynamic programming | counts or optimises without listing solutions; overlapping subproblems are cached |
| BFS | finds the *fewest* steps (Remove Invalid Parentheses); backtracking lists *all* answers |

**Learning path.** Subsets (record every node) → with duplicates → fixed size → with sum
(reuse, no reuse, both limits) → permutations (`used` array) → with duplicates → product of
choices (phone letters) → count constraints (parentheses) → bitmask feasibility. Then
constraint search: grid paths → string cuts → segment rules → N-Queens (sets, then
bitmasks) → Sudoku (most constrained cell first) → bucket filling → expressions with
precedence → BFS for minimum removals.

## 6.1 Subsets, combinations and permutations

Three axes decide the template: may an element be reused (`i` vs `i + 1`)? does order
matter (start at 0 with `used`, or at `start`)? can the input have duplicates (sort +
skip equal siblings)? Answer them before writing the loop.

### 6.1.1 [78. Subsets](https://leetcode.com/problems/subsets/) — ★★ · Medium

**Problem.** All subsets of distinct integers.  
**Signal.** Each element is in or out → backtracking (or iterate: double the list per element).  
*New here: The backtracking template; every node of the decision tree is an answer.*

**Fine print.** Ask before coding:
- Distinct values? → Yes.
- Any order? → Yes. Include the empty set? → Yes.

> 💡 **Key insight.** Each path from the root of the decision tree (choosing increasing indices) is a distinct subset, so record the path at *every* node.

**Brute force →** Iterate over all bitmasks 0..2ⁿ−1 → also O(n · 2ⁿ); backtracking is the template the harder variants build on.

**Steps**
1. Record a copy of the current path.
2. For each index `i` from `start`, append `nums[i]`.
3. Recurse with `i + 1`.
4. Pop to undo.

**Complexity**

- **Time:** O(n · 2ⁿ) — 2ⁿ subsets, each copied in O(n).
- **Space:** O(n) recursion (plus the output).

In [45]:
def subsets(nums: list[int]) -> list[list[int]]:
    """Power set."""
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int) -> None:
        # every node of the decision tree is a subset
        out.append(path.copy())
        for i in range(start, len(nums)):
            # choose, explore, un-choose
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return out


assert subsets([1, 2, 3]) == [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
print(len(subsets([1, 2, 3, 4])))

16


### 6.1.2 [90. Subsets II](https://leetcode.com/problems/subsets-ii/) — ★★ · Medium

**Problem.** All unique subsets when the input has duplicates.  
**Signal.** Sort, then skip equal siblings at the same depth.  
*New here: Duplicate skipping among siblings.*

**Fine print.** Ask before coding:
- Duplicates in the input? → Yes, so sort first to group equal values.
- Any order? → Yes.

> 💡 **Key insight.** At one level of the tree, picking the second of two equal values produces exactly the same subtrees as picking the first, so skip it.

**Brute force →** Generate all subsets and dedupe with a set of tuples → O(n · 2ⁿ) plus hashing. Skipping prevents the duplicates instead.

**Steps**
1. Sort.
2. Record the path.
3. Loop `i` from `start`; skip if `i > start` and `nums[i] == nums[i − 1]`.
4. Choose, recurse with `i + 1`, un-choose.

**Complexity**

- **Time:** O(n · 2ⁿ).
- **Space:** O(n).

⚠️ **Trap.** The condition is `i > start`, not `i > 0` — equal values *may* appear deeper in the same path.

In [46]:
def subsets_with_dup(nums: list[int]) -> list[list[int]]:
    """Unique subsets of a multiset."""
    # equal values become neighbours
    nums = sorted(nums)
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int) -> None:
        out.append(path.copy())
        for i in range(start, len(nums)):
            # same value as the previous sibling: same subtree, skip
            if i > start and nums[i] == nums[i - 1]:
                continue
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return out


assert subsets_with_dup([1, 2, 2]) == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
print(subsets_with_dup([1, 2, 2]))

[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]


### 6.1.3 [77. Combinations](https://leetcode.com/problems/combinations/) — ★★ · Medium

**Problem.** All combinations of `k` numbers from `1..n`.  
**Signal.** Subsets restricted to size `k`, with pruning.  
*New here: A completion condition plus range pruning.*

**Fine print.** Ask before coding:
- Combinations, not permutations? → Yes, order does not matter.
- Any order? → Yes.

> 💡 **Key insight.** If fewer numbers remain than slots to fill, no combination can finish, so the loop can stop early.

**Brute force →** Generate all subsets and keep size-k ones → O(n · 2ⁿ). Size-k backtracking with pruning → O(k · C(n, k)).

**Steps**
1. If the path has `k` numbers, record it.
2. Loop `i` from `start` up to the last value that still leaves room for the remaining slots.
3. Choose `i`, recurse with `i + 1`, un-choose.

**Complexity**

- **Time:** O(k · C(n, k)).
- **Space:** O(k).

In [47]:
def combine(n: int, k: int) -> list[list[int]]:
    """All k-combinations of 1..n."""
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int) -> None:
        if len(path) == k:
            out.append(path.copy())
            return
        # stop once too few numbers remain to fill the k slots
        for i in range(start, n - (k - len(path)) + 2):
            path.append(i)
            backtrack(i + 1)
            path.pop()

    backtrack(1)
    return out


assert combine(4, 2) == [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
assert combine(1, 1) == [[1]]
print(combine(4, 2))

[[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]


### 6.1.4 [39. Combination Sum](https://leetcode.com/problems/combination-sum/) — ★★ · Medium

**Problem.** Distinct candidates, unlimited reuse; all combinations summing to `target`.  
**Signal.** Reuse allowed → recurse with `i` (not `i + 1`); sorted input lets you prune.  
*New here: Reuse (`i`) and sorted pruning with `break`.*

**Fine print.** Ask before coding:
- Positive candidates? → Yes; with 0 or negatives the search would never end.
- Distinct candidates? → Yes.
- Combinations, not sequences? → Yes.

> 💡 **Key insight.** Recursing with the same index `i` allows reuse while still producing combinations in non-decreasing order (so no duplicates); sorting lets one oversized candidate end the whole loop.

**Brute force →** Try counts for every candidate → exponential without pruning. Sorted backtracking prunes most branches.

**Steps**
1. Sort the candidates.
2. Remaining 0 → record the path.
3. For each `i` from `start`: if the candidate exceeds the remaining sum, break.
4. Choose it, recurse with `i` (reuse) and the reduced sum, un-choose.

**Complexity**

- **Time:** Exponential, bounded by about `n^(target / min)`.
- **Space:** O(target / min) recursion.

In [48]:
def combination_sum(candidates: list[int], target: int) -> list[list[int]]:
    """Combinations with reuse summing to target."""
    candidates = sorted(candidates)
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int, remaining: int) -> None:
        if remaining == 0:
            out.append(path.copy())
            return
        for i in range(start, len(candidates)):
            # sorted: this and every later candidate is too big
            if candidates[i] > remaining:
                break
            path.append(candidates[i])
            # i, not i + 1: the same candidate may be used again
            backtrack(i, remaining - candidates[i])
            path.pop()

    backtrack(0, target)
    return out


assert combination_sum([2, 3, 6, 7], 7) == [[2, 2, 3], [7]]
assert combination_sum([2, 3, 5], 8) == [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
assert combination_sum([2], 1) == []
print(combination_sum([2, 3, 6, 7], 7))

[[2, 2, 3], [7]]


### 6.1.5 [40. Combination Sum II](https://leetcode.com/problems/combination-sum-ii/) — ★★ · Medium

**Problem.** Candidates may repeat; each used at most once; unique combinations summing to `target`.  
**Signal.** No reuse (`i + 1`) + duplicate skipping (Subsets II rule).  
*New here: Combining no-reuse, duplicate skipping and sum pruning.*

**Fine print.** Ask before coding:
- Can each element be used once? → Yes, each index once.
- Unique combinations only? → Yes.

> 💡 **Key insight.** It is Subsets II with a sum condition: skip equal siblings to avoid duplicate combinations, and break once a candidate is too big.

**Brute force →** All subsets + set of sorted tuples → O(n · 2ⁿ). The combined rules prune heavily.

**Steps**
1. Sort.
2. Remaining 0 → record.
3. Skip `i > start` with the same value as `i − 1`; break if the candidate exceeds the remainder.
4. Choose, recurse with `i + 1`, un-choose.

**Complexity**

- **Time:** O(2ⁿ) worst case.
- **Space:** O(n).

In [49]:
def combination_sum2(candidates: list[int], target: int) -> list[list[int]]:
    """Unique single-use combinations summing to target."""
    candidates = sorted(candidates)
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int, remaining: int) -> None:
        if remaining == 0:
            out.append(path.copy())
            return
        for i in range(start, len(candidates)):
            # equal sibling would repeat a combination
            if i > start and candidates[i] == candidates[i - 1]:
                continue
            if candidates[i] > remaining:
                break
            path.append(candidates[i])
            # i + 1: each candidate used at most once
            backtrack(i + 1, remaining - candidates[i])
            path.pop()

    backtrack(0, target)
    return out


assert combination_sum2([10, 1, 2, 7, 6, 1, 5], 8) == [
    [1, 1, 6],
    [1, 2, 5],
    [1, 7],
    [2, 6],
]
assert combination_sum2([2, 5, 2, 1, 2], 5) == [[1, 2, 2], [5]]
print(combination_sum2([10, 1, 2, 7, 6, 1, 5], 8))

[[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]


### 6.1.6 [216. Combination Sum III](https://leetcode.com/problems/combination-sum-iii/) — ★ · Medium

**Problem.** All combinations of `k` distinct digits 1–9 summing to `n`.  
**Signal.** Combinations with both a size and a sum constraint.  
*New here: Two stopping conditions at once (size and sum).*

**Fine print.** Ask before coding:
- Digits 1–9, each at most once? → Yes.
- Exactly `k` numbers? → Yes.

> 💡 **Key insight.** Stop at exactly `k` digits, and record only if the sum is used up exactly.

**Brute force →** Check all 2⁹ subsets of digits → tiny either way; backtracking practises the template.

**Steps**
1. Path has `k` digits → record if remaining is 0, then return.
2. Loop digits from `start` to 9; break if a digit exceeds the remainder.
3. Choose, recurse with `d + 1`, un-choose.

**Complexity**

- **Time:** O(C(9, k)).
- **Space:** O(k).

In [50]:
def combination_sum3(k: int, n: int) -> list[list[int]]:
    """k distinct digits summing to n."""
    out: list[list[int]] = []
    path: list[int] = []

    def backtrack(start: int, remaining: int) -> None:
        # size reached: valid only if the sum is exact
        if len(path) == k:
            if remaining == 0:
                out.append(path.copy())
            return
        for d in range(start, 10):
            if d > remaining:
                break
            path.append(d)
            backtrack(d + 1, remaining - d)
            path.pop()

    backtrack(1, n)
    return out


assert combination_sum3(3, 7) == [[1, 2, 4]]
assert combination_sum3(3, 9) == [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
assert combination_sum3(4, 1) == []
print(combination_sum3(3, 9))

[[1, 2, 6], [1, 3, 5], [2, 3, 4]]


### 6.1.7 [46. Permutations](https://leetcode.com/problems/permutations/) — ★★ · Medium

**Problem.** All permutations of distinct integers.  
**Signal.** Order matters → loop from 0 each level, skip used elements.  
*New here: Order matters: loop from 0 with a `used` array.*

**Fine print.** Ask before coding:
- Distinct values? → Yes.
- Any order? → Yes.

> 💡 **Key insight.** Every position can take any element not already placed, so each level loops over all elements and skips the used ones.

**Brute force →** `itertools.permutations` does it; writing it shows the `used` pattern needed for the duplicate version.

**Steps**
1. Path length `n` → record.
2. For each index not yet used, mark it and append its value.
3. Recurse.
4. Pop and unmark.

**Complexity**

- **Time:** O(n · n!) — n! permutations, each copied in O(n).
- **Space:** O(n).

In [51]:
def permute(nums: list[int]) -> list[list[int]]:
    """All orderings of nums."""
    out: list[list[int]] = []
    path: list[int] = []
    used = [False] * len(nums)

    def backtrack() -> None:
        if len(path) == len(nums):
            out.append(path.copy())
            return
        # any unused element may take the next position
        for i, x in enumerate(nums):
            if not used[i]:
                used[i] = True
                path.append(x)
                backtrack()
                path.pop()
                used[i] = False

    backtrack()
    return out


assert permute([1, 2, 3]) == [
    [1, 2, 3],
    [1, 3, 2],
    [2, 1, 3],
    [2, 3, 1],
    [3, 1, 2],
    [3, 2, 1],
]
assert permute([1]) == [[1]]
print(len(permute([1, 2, 3, 4])))

24


### 6.1.8 [47. Permutations II](https://leetcode.com/problems/permutations-ii/) — ★ · Medium

**Problem.** All unique permutations when the input has duplicates.  
**Signal.** Sort + skip a value equal to the previous one if the previous copy is unused.  
*New here: The duplicate rule for permutations (uses `used[i − 1]`).*

**Fine print.** Ask before coding:
- Duplicates? → Yes, so sort and skip equal values that have not been used.
- Any order? → Yes.

> 💡 **Key insight.** Force equal values to be placed in their original left-to-right order; then each distinct arrangement is generated exactly once.

**Brute force →** All n! permutations into a set → wasteful with many duplicates. The skip rule generates only unique ones.

**Steps**
1. Sort.
2. Skip index `i` if it is used.
3. Also skip if `nums[i] == nums[i − 1]` and `i − 1` is *not* used (its earlier copy must go first).
4. Choose, recurse, un-choose.

**Complexity**

- **Time:** O(n · n!) worst case.
- **Space:** O(n).

In [52]:
def permute_unique(nums: list[int]) -> list[list[int]]:
    """Unique orderings of a multiset."""
    nums = sorted(nums)
    out: list[list[int]] = []
    path: list[int] = []
    used = [False] * len(nums)

    def backtrack() -> None:
        if len(path) == len(nums):
            out.append(path.copy())
            return
        for i, x in enumerate(nums):
            # equal values must be taken left to right
            if used[i] or (i and nums[i] == nums[i - 1] and not used[i - 1]):
                continue
            used[i] = True
            path.append(x)
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return out


assert permute_unique([1, 1, 2]) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
assert len(permute_unique([1, 2, 3])) == 6
print(permute_unique([1, 1, 2]))

[[1, 1, 2], [1, 2, 1], [2, 1, 1]]


### 6.1.9 [17. Letter Combinations of a Phone Number](https://leetcode.com/problems/letter-combinations-of-a-phone-number/) — ★★ · Medium

**Problem.** All letter strings a phone digit string could represent.  
**Signal.** One choice per position from a fixed mapping → backtracking (Cartesian product).  
*New here: One independent choice per position (a Cartesian product).*

**Fine print.** Ask before coding:
- Digits 2–9 only? → Yes.
- Empty input? → `[]`, not `[""]`.

> 💡 **Key insight.** Each digit independently picks one of its letters, so recurse position by position over its options.

**Brute force →** `itertools.product` gives the same; the recursion is the general form.

**Steps**
1. Empty input → `[]`.
2. At position `i`, try each letter of `digits[i]`.
3. Recurse with `i + 1` and the extended string.
4. At the end of the digits, record the string.

**Complexity**

- **Time:** O(4ⁿ · n).
- **Space:** O(n) recursion.

In [53]:
PHONE = {
    "2": "abc",
    "3": "def",
    "4": "ghi",
    "5": "jkl",
    "6": "mno",
    "7": "pqrs",
    "8": "tuv",
    "9": "wxyz",
}


def letter_combinations(digits: str) -> list[str]:
    """Letter strings for a digit string."""
    if not digits:
        return []
    out: list[str] = []

    def backtrack(i: int, path: str) -> None:
        if i == len(digits):
            out.append(path)
            return
        # each letter of this digit is one branch
        for ch in PHONE[digits[i]]:
            backtrack(i + 1, path + ch)

    backtrack(0, "")
    return out


assert letter_combinations("23") == [
    "ad",
    "ae",
    "af",
    "bd",
    "be",
    "bf",
    "cd",
    "ce",
    "cf",
]
assert letter_combinations("") == []
print(letter_combinations("23"))

['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


### 6.1.10 [22. Generate Parentheses](https://leetcode.com/problems/generate-parentheses/) — ★★ · Medium

**Problem.** All well-formed strings of `n` pairs of parentheses.  
**Signal.** Choices constrained by counts → backtrack with `open` and `close` counters.  
*New here: Validity enforced by counters, so invalid prefixes are never built.*

**Fine print.** Ask before coding:
- Any order? → Yes.
- Return all of them, not just the count? → All of them.

> 💡 **Key insight.** A prefix stays valid as long as you never use more than `n` opens and never close more than you have opened.

**Brute force →** Generate all 2²ⁿ strings and check each → O(2²ⁿ · n). Counter-guided backtracking only builds valid ones.

**Steps**
1. Length `2n` → record.
2. If `open < n`, add `(`.
3. If `close < open`, add `)`.

**Complexity**

- **Time:** O(4ⁿ / √n) — the Catalan number of results.
- **Space:** O(n).

In [54]:
def generate_parenthesis(n: int) -> list[str]:
    """All balanced strings of n pairs."""
    out: list[str] = []

    def backtrack(path: str, open_count: int, close_count: int) -> None:
        if len(path) == 2 * n:
            out.append(path)
            return
        # an opener is allowed while some remain
        if open_count < n:
            backtrack(path + "(", open_count + 1, close_count)
        # a closer is allowed only if it has an opener to match
        if close_count < open_count:
            backtrack(path + ")", open_count, close_count + 1)

    backtrack("", 0, 0)
    return out


assert generate_parenthesis(3) == ["((()))", "(()())", "(())()", "()(())", "()()()"]
assert generate_parenthesis(1) == ["()"]
print(generate_parenthesis(3))

['((()))', '(()())', '(())()', '()(())', '()()()']


### 6.1.11 [1239. Maximum Length of a Concatenated String with Unique Characters](https://leetcode.com/problems/maximum-length-of-a-concatenated-string-with-unique-characters/) — ★ · Medium

**Problem.** Choose a subsequence of strings whose concatenation has all-unique characters; maximise its length.  
**Signal.** Subset enumeration with a feasibility check → backtracking with bitmasks.  
*New here: Bitmasks as fast sets of letters.*

**Fine print.** Ask before coding:
- Can a string contain a repeated character? → Yes, and then it can never be used.
- Lowercase only? → Yes, so a 26-bit mask works.

> 💡 **Key insight.** Each string is a 26-bit mask; two strings can be combined iff their masks share no bit (`a & b == 0`).

**Brute force →** Try all subsets with set unions → O(2ⁿ · 26). Bitmasks make each check O(1).

**Steps**
1. Convert strings to masks; drop strings with repeated letters.
2. `backtrack(i, used)`: the current length is the number of set bits.
3. For each later mask not overlapping `used`, recurse with the union.
4. Return the best length found.

**Complexity**

- **Time:** O(2ⁿ).
- **Space:** O(n).

In [55]:
def max_length(arr: list[str]) -> int:
    """Longest unique-char concatenation length."""
    # one bit per letter; skip strings with repeats
    masks = []
    for s in arr:
        m = 0
        for ch in s:
            m |= 1 << (ord(ch) - 97)
        if bin(m).count("1") == len(s):
            masks.append(m)

    def backtrack(i: int, used: int) -> int:
        best = bin(used).count("1")
        for j in range(i, len(masks)):
            # no shared letters: safe to add
            if not used & masks[j]:
                best = max(best, backtrack(j + 1, used | masks[j]))
        return best

    return backtrack(0, 0)


assert max_length(["un", "iq", "ue"]) == 4
assert max_length(["cha", "r", "act", "ers"]) == 6
assert max_length(["abcdefghijklmnopqrstuvwxyz"]) == 26
print(max_length(["cha", "r", "act", "ers"]))

6


## 6.2 Search with constraints

Here the choices are constrained by a board, a string or a running value. The skill is
checking validity in O(1) (sets, bitmasks, counters) and pruning *before* recursing.

### 6.2.1 [79. Word Search](https://leetcode.com/problems/word-search/) — ★★★ · Medium

**Problem.** Does the word exist in the grid as a path of adjacent cells (no reuse)?  
**Signal.** Path search with "no revisit" → DFS with temporary marking and backtracking.  
*New here: Backtracking on a grid: mark a cell, explore, then *unmark* it.*

**Fine print.** Ask before coding:
- Can a cell be reused? → No.
- Case-sensitive? → Yes.
- Can I modify the board temporarily? → Yes, as long as it is restored.

> 💡 **Key insight.** A cell may be reused by a *different* path, so it is marked only while it is on the current path and unmarked on the way back.

**Brute force →** Enumerate all paths of length L from every cell without pruning → same worst case; matching letter by letter prunes almost every branch.

**Steps**
1. From each cell, DFS with index `i` into the word.
2. Out of bounds or letter mismatch → false; `i == len(word)` → true.
3. Mark the cell, try the 4 neighbours with `i + 1`.
4. Unmark the cell and return the result.

**Complexity**

- **Time:** O(cells · 3^L) — after the first step each cell has at most 3 new directions.
- **Space:** O(L) recursion.

In [56]:
def exist(board: list[list[str]], word: str) -> bool:
    """True if word can be traced in the grid."""
    rows, cols = len(board), len(board[0])

    def dfs(r: int, c: int, i: int) -> bool:
        if i == len(word):
            return True
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False
        # mark as used on the current path
        board[r][c] = "#"
        found = any(dfs(r + dr, c + dc, i + 1) for dr, dc in DIRS)
        # free the cell for other paths
        board[r][c] = word[i]
        return found

    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))


b = [list("ABCE"), list("SFCS"), list("ADEE")]
assert exist(b, "ABCCED") and exist(b, "SEE") and not exist(b, "ABCB")
print("ok")

ok


### 6.2.2 [131. Palindrome Partitioning](https://leetcode.com/problems/palindrome-partitioning/) — ★★ · Medium

**Problem.** All ways to split `s` into palindromic pieces.  
**Signal.** Choose the next cut; only continue if the piece is a palindrome.  
*New here: Choices are "where to cut next".*

**Fine print.** Ask before coding:
- Every way, not just one? → Yes.
- Lowercase only? → Yes.

> 💡 **Key insight.** A partition is a sequence of cut points; at each position, only cuts that produce a palindromic piece can lead to a valid answer.

**Brute force →** Try all 2ⁿ⁻¹ cut patterns and check every piece → O(n · 2ⁿ) with no pruning. Checking each piece before recursing prunes early.

**Steps**
1. At the end of the string, record the pieces.
2. For each end position, take `s[start:end]`.
3. If it is a palindrome, append it, recurse from `end`, pop.

**Complexity**

- **Time:** O(n · 2ⁿ).
- **Space:** O(n).

In [57]:
def partition_palindromes(s: str) -> list[list[str]]:
    """Every palindromic partition of s."""
    out: list[list[str]] = []
    path: list[str] = []

    def backtrack(start: int) -> None:
        if start == len(s):
            out.append(path.copy())
            return
        # try every next cut position
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            # only palindromic pieces may continue
            if piece == piece[::-1]:
                path.append(piece)
                backtrack(end)
                path.pop()

    backtrack(0)
    return out


assert partition_palindromes("aab") == [["a", "a", "b"], ["aa", "b"]]
assert partition_palindromes("a") == [["a"]]
print(partition_palindromes("aab"))

[['a', 'a', 'b'], ['aa', 'b']]


### 6.2.3 [93. Restore IP Addresses](https://leetcode.com/problems/restore-ip-addresses/) — ★ · Medium

**Problem.** All valid IPv4 addresses formed by inserting three dots into a digit string.  
**Signal.** Four segments, each 1–3 digits, value ≤ 255, no leading zeros.  
*New here: Pruning on remaining length (too few or too many digits left).*

**Fine print.** Ask before coding:
- Are leading zeros allowed? → No, except `"0"` itself.
- Is each part 0–255? → Yes.
- Any order? → Yes.

> 💡 **Key insight.** With `p` segments left, the remaining digits must number between `p` and `3p`; otherwise stop immediately.

**Brute force →** Try all 3 dot positions → O(n³) placements, each validated; backtracking with pruning is the same idea organised segment by segment.

**Steps**
1. Four parts and all digits used → record.
2. If the remaining digits cannot fit the remaining parts, return.
3. Try segment lengths 1–3; stop on leading zeros or a value > 255.
4. Recurse with the segment added.

**Complexity**

- **Time:** O(3⁴) — at most 81 branches.
- **Space:** O(1) besides the output.

In [58]:
def restore_ip_addresses(s: str) -> list[str]:
    """All valid IPs from the digit string."""
    out: list[str] = []

    def backtrack(start: int, parts: list[str]) -> None:
        if len(parts) == 4:
            if start == len(s):
                out.append(".".join(parts))
            return
        # remaining digits must fit 1..3 per remaining part
        remaining = len(s) - start
        if not 4 - len(parts) <= remaining <= 3 * (4 - len(parts)):
            return
        for length in range(1, 4):
            seg = s[start : start + length]
            # too short, leading zero, or over 255: longer segments fail too
            if len(seg) < length or (seg[0] == "0" and length > 1) or int(seg) > 255:
                break
            backtrack(start + length, parts + [seg])

    backtrack(0, [])
    return out


assert restore_ip_addresses("25525511135") == ["255.255.11.135", "255.255.111.35"]
assert restore_ip_addresses("0000") == ["0.0.0.0"]
assert restore_ip_addresses("101023") == [
    "1.0.10.23",
    "1.0.102.3",
    "10.1.0.23",
    "10.10.2.3",
    "101.0.2.3",
]
print(restore_ip_addresses("25525511135"))

['255.255.11.135', '255.255.111.35']


### 6.2.4 [51. N-Queens](https://leetcode.com/problems/n-queens/) — ★★ · Hard

**Problem.** All placements of `n` non-attacking queens on an `n × n` board.  
**Signal.** One queen per row; track used columns and both diagonals in sets.  
*New here: O(1) attack checks with column and diagonal sets.*

**Fine print.** Ask before coding:
- Return the boards as strings of `Q` and `.`? → Yes.
- `n ≤ 9`? → Yes.

> 💡 **Key insight.** Cells on one diagonal share `r − c` and on one anti-diagonal share `r + c`, so three sets answer "is this square attacked?" instantly.

**Brute force →** Place n queens anywhere and check → C(n², n). One queen per row with set checks → O(n!).

**Steps**
1. Place queens row by row.
2. Skip column `c` if `c`, `r − c` or `r + c` is used.
3. Add to the three sets, recurse to the next row.
4. Remove from the sets; at row `n`, build the board.

**Complexity**

- **Time:** O(n!).
- **Space:** O(n).

In [59]:
def solve_n_queens(n: int) -> list[list[str]]:
    """All N-Queens boards."""
    out: list[list[str]] = []
    # attacked columns, diagonals (r - c) and anti-diagonals (r + c)
    cols: set[int] = set()
    diag: set[int] = set()
    anti: set[int] = set()
    queens: list[int] = []

    def backtrack(r: int) -> None:
        if r == n:
            out.append(["." * c + "Q" + "." * (n - c - 1) for c in queens])
            return
        for c in range(n):
            if c in cols or r - c in diag or r + c in anti:
                continue
            # place the queen
            cols.add(c)
            diag.add(r - c)
            anti.add(r + c)
            queens.append(c)
            backtrack(r + 1)
            # remove the queen
            queens.pop()
            cols.remove(c)
            diag.remove(r - c)
            anti.remove(r + c)

    backtrack(0)
    return out


assert solve_n_queens(4) == [
    [".Q..", "...Q", "Q...", "..Q."],
    ["..Q.", "Q...", "...Q", ".Q.."],
]
assert len(solve_n_queens(1)) == 1
print(len(solve_n_queens(8)))

92


### 6.2.5 [52. N-Queens II](https://leetcode.com/problems/n-queens-ii/) — ★★ · Hard

**Problem.** Count the N-Queens solutions.  
**Signal.** Same search, count instead of building boards; bitmasks make it fast.  
*New here: Bitmask state and lowest-set-bit iteration.*

**Fine print.** Ask before coding:
- Only the count? → Yes.

> 💡 **Key insight.** Diagonal attacks shift one column per row, so the diagonal masks can be shifted left/right instead of storing `r − c` values.

**Brute force →** The set-based N-Queens counting boards → same complexity, larger constants. Bitmasks are much faster.

**Steps**
1. `cols`, `diag`, `anti` are bitmasks of attacked columns for the current row.
2. Free squares = `~(cols | diag | anti) & full`.
3. Take the lowest free bit, recurse with updated masks (diag shifted left, anti shifted right).
4. All columns filled → count 1.

**Complexity**

- **Time:** O(n!) with tiny constants.
- **Space:** O(n) recursion.

In [60]:
def total_n_queens(n: int) -> int:
    """Number of N-Queens solutions."""
    full = (1 << n) - 1

    def count(cols: int, diag: int, anti: int) -> int:
        # every column holds a queen
        if cols == full:
            return 1
        total = 0
        # squares in this row not attacked by anything
        free = ~(cols | diag | anti) & full
        while free:
            # lowest free square
            bit = free & -free
            free -= bit
            # diagonals move one column per row
            total += count(cols | bit, ((diag | bit) << 1) & full, (anti | bit) >> 1)
        return total

    return count(0, 0, 0)


assert [total_n_queens(n) for n in range(1, 9)] == [1, 0, 0, 2, 10, 4, 40, 92]
print(total_n_queens(8))

92


### 6.2.6 [37. Sudoku Solver](https://leetcode.com/problems/sudoku-solver/) — ★★ · Hard

**Problem.** Fill a 9×9 Sudoku in place.  
**Signal.** Constraint search: try digits in empty cells with row/column/box sets.  
*New here: Choosing the most constrained cell first (MRV heuristic).*

**Fine print.** Ask before coding:
- Exactly one solution? → Yes.
- In place? → Yes.

> 💡 **Key insight.** Filling the empty cell with the fewest legal digits first keeps the search tree tiny, because wrong choices are discovered almost immediately.

**Brute force →** Fill cells in reading order → works, but can explore many more branches. MRV makes typical puzzles instant.

**Steps**
1. Record used digits per row, column and box; list the empty cells.
2. Pick the empty cell with the fewest options.
3. Try each option: place it, update the sets, recurse.
4. On failure, undo and put the cell back in the empty list.

**Complexity**

- **Time:** Exponential worst case, fast in practice.
- **Space:** O(81).

In [61]:
def solve_sudoku(board: list[list[str]]) -> None:
    """Solve the puzzle in place."""
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empty = []
    for r in range(9):
        for c in range(9):
            d = board[r][c]
            if d == ".":
                empty.append((r, c))
            else:
                rows[r].add(d)
                cols[c].add(d)
                boxes[r // 3 * 3 + c // 3].add(d)

    def options(r: int, c: int) -> set[str]:
        """Digits not yet used in this cell's row, column or box."""
        return set("123456789") - rows[r] - cols[c] - boxes[r // 3 * 3 + c // 3]

    def backtrack() -> bool:
        if not empty:
            return True
        # most constrained cell first
        best = min(range(len(empty)), key=lambda i: len(options(*empty[i])))
        empty[best], empty[-1] = empty[-1], empty[best]
        r, c = empty.pop()
        for d in options(r, c):
            # place d
            board[r][c] = d
            rows[r].add(d)
            cols[c].add(d)
            boxes[r // 3 * 3 + c // 3].add(d)
            if backtrack():
                return True
            # undo d
            rows[r].remove(d)
            cols[c].remove(d)
            boxes[r // 3 * 3 + c // 3].remove(d)
        board[r][c] = "."
        empty.append((r, c))
        return False

    backtrack()


puzzle = [
    "53..7....",
    "6..195...",
    ".98....6.",
    "8...6...3",
    "4..8.3..1",
    "7...2...6",
    ".6....28.",
    "...419..5",
    "....8..79",
]
board = [list(row) for row in puzzle]
solve_sudoku(board)
assert "".join(board[0]) == "534678912" and "".join(board[8]) == "345286179"
print("\n".join("".join(r) for r in board))

534678912
672195348
198342567
859761423
426853791
713924856
961537284
287419635
345286179


### 6.2.7 [698. Partition to K Equal Sum Subsets](https://leetcode.com/problems/partition-to-k-equal-sum-subsets/) — ★ · Medium

**Problem.** Can the array be split into `k` subsets with equal sums?  
**Signal.** Fill k buckets with backtracking; sort descending and skip symmetric states.  
*New here: Symmetry pruning: empty buckets are interchangeable.*

**Fine print.** Ask before coding:
- Positive values? → Yes.
- Must every element be used? → Yes.
- `k ≤ n ≤ 16`? → Yes, so bitmask DP is an option.

> 💡 **Key insight.** If a number fails in one empty bucket, it fails in every empty bucket, so only one empty bucket needs to be tried.

**Brute force →** Assign each number to any of k buckets → O(kⁿ). Sorting descending + symmetry pruning cuts this drastically.

**Steps**
1. Sum must divide by k; the largest number must fit in `target`.
2. Sort descending (big items fail fast).
3. For each number, try each bucket with room; recurse; undo.
4. If the bucket was empty after undoing, stop trying other buckets.

**Complexity**

- **Time:** O(kⁿ) worst case, heavily pruned.
- **Space:** O(n + k).

In [62]:
def can_partition_k_subsets(nums: list[int], k: int) -> bool:
    """True if nums splits into k equal-sum groups."""
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    # largest first: failures show up early
    nums = sorted(nums, reverse=True)
    if nums[0] > target:
        return False
    buckets = [0] * k

    def backtrack(i: int) -> bool:
        if i == len(nums):
            return True
        for b in range(k):
            if buckets[b] + nums[i] <= target:
                buckets[b] += nums[i]
                if backtrack(i + 1):
                    return True
                buckets[b] -= nums[i]
            # empty buckets are identical: one failure covers them all
            if buckets[b] == 0:
                break
        return False

    return backtrack(0)


assert can_partition_k_subsets([4, 3, 2, 3, 5, 2, 1], 4)
assert not can_partition_k_subsets([1, 2, 3, 4], 3)
print("ok")

ok


### 6.2.8 [473. Matchsticks to Square](https://leetcode.com/problems/matchsticks-to-square/) — ★ · Medium

**Problem.** Use all matchsticks to form a square.  
**Signal.** Partition into 4 equal-sum subsets.  
*New here: Recognising a known problem in disguise.*

**Fine print.** Ask before coding:
- Must all matchsticks be used without breaking them? → Yes.
- `n ≤ 15`? → Yes.

> 💡 **Key insight.** Four sides of equal length = partition into 4 equal-sum subsets.

**Brute force →** Assign each stick to a side → 4ⁿ; reuse the pruned bucket search.

**Steps**
1. Fewer than 4 sticks → false.
2. Call the k-bucket partition with `k = 4`.

**Complexity**

- **Time:** O(4ⁿ) worst case, pruned.
- **Space:** O(n).

In [63]:
def makesquare(matchsticks: list[int]) -> bool:
    """True if the sticks form a square."""
    # four equal sides = four equal-sum buckets
    return len(matchsticks) >= 4 and can_partition_k_subsets(matchsticks, 4)


assert makesquare([1, 1, 2, 2, 2])
assert not makesquare([3, 3, 3, 3, 4])
print("ok")

ok


### 6.2.9 [282. Expression Add Operators](https://leetcode.com/problems/expression-add-operators/) — ★ · Hard

**Problem.** Insert `+`, `−`, `*` between digits of `num` so the expression equals `target`.  
**Signal.** Backtracking that tracks the last multiplicative term to undo it for `*`.  
*New here: Carrying extra state (`last` term) to handle operator precedence.*

**Fine print.** Ask before coding:
- Are operands with leading zeros allowed? → No, `"05"` is invalid.
- Is there operator precedence? → Yes, `*` binds tighter.
- Can intermediate values exceed 32 bits? → Yes.

> 💡 **Key insight.** When `*` follows, the previous term must be multiplied, not the whole total: `value − last + last × x` swaps the last term for its product.

**Brute force →** Build every expression string and `eval` it → O(4ⁿ · n) and slow. Carrying `value` and `last` evaluates on the fly.

**Steps**
1. Choose the next operand `num[i:j]` (no leading zeros).
2. First operand: start `value = last = x`.
3. `+x`: value + x, last x. `−x`: value − x, last −x.
4. `*x`: value − last + last·x, last last·x. Record when all digits are used and value equals target.

**Complexity**

- **Time:** O(4ⁿ) — 3 operators or "extend the number" at each gap.
- **Space:** O(n) recursion.

In [64]:
def add_operators(num: str, target: int) -> list[str]:
    """Expressions over num that evaluate to target."""
    out: list[str] = []

    def backtrack(i: int, expr: str, value: int, last: int) -> None:
        if i == len(num):
            if value == target:
                out.append(expr)
            return
        for j in range(i + 1, len(num) + 1):
            part = num[i:j]
            # multi-digit operands may not start with 0
            if len(part) > 1 and part[0] == "0":
                break
            x = int(part)
            if i == 0:
                backtrack(j, part, x, x)
            else:
                backtrack(j, f"{expr}+{part}", value + x, x)
                backtrack(j, f"{expr}-{part}", value - x, -x)
                # undo the last term, then apply it multiplied
                backtrack(j, f"{expr}*{part}", value - last + last * x, last * x)

    backtrack(0, "", 0, 0)
    return out


assert sorted(add_operators("123", 6)) == ["1*2*3", "1+2+3"]
assert sorted(add_operators("232", 8)) == ["2*3+2", "2+3*2"]
assert add_operators("3456237490", 9191) == []
assert sorted(add_operators("105", 5)) == ["1*0+5", "10-5"]
print(add_operators("123", 6))

['1+2+3', '1*2*3']


### 6.2.10 [301. Remove Invalid Parentheses](https://leetcode.com/problems/remove-invalid-parentheses/) — ★ · Hard

**Problem.** Remove the minimum number of parentheses to make the string valid; return all results.  
**Signal.** Minimum removals → BFS by number of removals; first level with a valid string wins.  
*New here: BFS instead of DFS when "minimum" and "all" are both required.*

**Fine print.** Ask before coding:
- Remove the minimum number only? → Yes.
- Unique results? → Yes.
- Keep letters? → Yes.

> 💡 **Key insight.** BFS level `d` holds every string with `d` removals, so the first level that contains a valid string has the minimum removals, and all valid strings there are the answer.

**Brute force →** DFS over all removal subsets and keep the shortest valid ones → explores far deeper. Level BFS stops at the first valid level.

**Steps**
1. Level 0 = `{s}`.
2. If any string in the level is valid, return them all (sorted).
3. Otherwise build the next level by deleting one parenthesis at every position (a set dedupes).
4. Repeat.

**Complexity**

- **Time:** Exponential worst case, fine for n ≤ 25.
- **Space:** Size of the largest level.

In [65]:
def remove_invalid_parentheses(s: str) -> list[str]:
    """All valid strings after minimum removals."""

    def valid(t: str) -> bool:
        """Balanced and never closes more than it opened."""
        balance = 0
        for ch in t:
            balance += ch == "("
            balance -= ch == ")"
            if balance < 0:
                return False
        return balance == 0

    # all strings with the same number of removals
    level = {s}
    while True:
        good = [t for t in level if valid(t)]
        # the first level with any valid string is the minimum
        if good:
            return sorted(good)
        # one more removal, deduplicated
        level = {
            t[:i] + t[i + 1 :] for t in level for i in range(len(t)) if t[i] in "()"
        }


assert remove_invalid_parentheses("()())()") == ["(())()", "()()()"]
assert remove_invalid_parentheses("(a)())()") == ["(a())()", "(a)()()"]
assert remove_invalid_parentheses(")(") == [""]
print(remove_invalid_parentheses("()())()"))

['(())()', '()()()']


---
# Summary

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Grid DFS | islands, regions, flood fill | mark visited before recursing | O(mn) |
| Reverse search | border-connected, ocean flow | start from the target side and invert | O(mn) |
| Multi-source BFS | distance to nearest X, spreading | enqueue all sources at distance 0 | O(mn) |
| BFS on states | word ladder, lock, buses | generate neighbours on the fly | O(states) |
| Topological sort | prerequisites, ordering, alien alphabet | Kahn: peel in-degree 0 | O(V + E) |
| Leaf peeling | tree centres | repeatedly remove degree-1 nodes | O(n) |
| Union–find | redundant edge, merging groups, equations | `find` with compression, union by size | ~O(1) |
| Dijkstra | non-negative weighted shortest path | min-heap, skip stale entries | O(E log V) |
| Bottleneck Dijkstra | minimise max edge on a path | `max` instead of `+` | O(E log V) |
| Bellman–Ford (k rounds) | at most k stops | relax all edges from the previous round's copy | O(kE) |
| MST (Prim) | connect all points cheaply | grow tree by cheapest outside node | O(n²) dense |
| Eulerian path | use every edge once | Hierholzer, build route in reverse | O(E log E) |
| Backtracking | all subsets / permutations / boards | choose → recurse → un-choose, prune early | exponential |